In [ ]:

import os
import pandas as pd

In [ ]:
folder_path = 'path_to_orchid_folder'

In [ ]:
all_files = os.listdir(folder_path)
print(all_files)

In [ ]:
folder_path = 'path_to_orchid_folder'
print(f"Verifying directory path: {folder_path}")
try:
    all_files = os.listdir(folder_path)
    print("Files in the directory:")
    print(all_files)
except FileNotFoundError:
    print(f"Error: The directory '{folder_path}' was not found.")
    print("Please ensure the path is correct and the folder exists in your Google Drive.")

In [ ]:
folder_path = 'path_to_orchid_folder'
print(f"Attempting to list files in: {folder_path}")

# Check if the immediate parent directory exists
parent_directory = os.path.dirname(folder_path)
if not os.path.exists(parent_directory):
    print(f"Error: The parent directory '{parent_directory}' was not found.")
    print("Please ensure that 'googledrive' exists directly under 'MyDrive'.")
else:
    try:
        all_files = os.listdir(folder_path)
        print("Files in the directory:")
        print(all_files)
    except FileNotFoundError:
        print(f"Error: The directory '{folder_path}' was not found.")
        print("Please ensure the path is correct and the 'ORCHID' folder exists within 'googledrive'.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
csv_files = [f for f in all_files if f.endswith('.csv')]
print("CSV files found:")
print(csv_files)

In [ ]:
dataframes = {}

for csv_file in csv_files:
    file_path = os.path.join(folder_path, csv_file)
    try:
        df_name = os.path.splitext(csv_file)[0] # Use filename without extension as key
        dataframes[df_name] = pd.read_csv(file_path)
        print(f"Successfully read {csv_file}")
    except Exception as e:
        print(f"Error reading {csv_file}: {e}")

In [ ]:
merged_df = dataframes['OPOReferrals']

In [ ]:
prepared_dataframes = {}

for df_name, df in dataframes.items():
    if df_name != 'OPOReferrals':
        print(f"\n--- Inspecting DataFrame: {df_name} ---")
        display(df.head())
        display(df.info())
        # Placeholder for strategy determination and transformation
        # The actual transformation will be implemented based on inspection

        # Example: For event data, consider aggregating or selecting key events
        if 'time_event' in df.columns or 'time_event_start' in df.columns:
            print(f"Strategy for {df_name}: Event data detected. Will consider aggregation or selecting key events per patient_id.")
            # Example placeholder for aggregation strategy: count events per patient
            # patient_event_counts = df['patient_id'].value_counts().reset_index()
            # patient_event_counts.columns = ['patient_id', f'{df_name}_event_count']
            # prepared_dataframes[df_name] = patient_event_counts

        # Example: For dataframes with a single entry per patient or clear identifying columns
        # if 'patient_id' in df.columns and len(df['patient_id'].unique()) < len(df):
        #     print(f"Strategy for {df_name}: Potential for direct merging or selecting unique patient entries.")
            # Example placeholder: Select distinct patient_id entries if applicable
            # unique_patient_data = df.drop_duplicates(subset=['patient_id'])
            # prepared_dataframes[df_name] = unique_patient_data

        # If no specific strategy is immediately apparent, keep the original for now and revisit
        # else:
        #     print(f"Strategy for {df_name}: Structure not immediately clear for aggregation/selection. Keeping original for now.")
        #     prepared_dataframes[df_name] = df.copy() # Use a copy to avoid modifying original dataframes

In [ ]:
prepared_dataframes = {}

# Define aggregation strategies for event dataframes
aggregation_strategies = {
    'SerologyEvents': lambda df: df.groupby('patient_id')['result'].apply(lambda x: ', '.join(x.unique())).reset_index(name='serology_results'),
    'CBCEvents': lambda df: df.groupby('patient_id').agg(
        cbc_wbc_mean=('value', lambda x: x[df.loc[x.index, 'cbc_name'] == 'WBC'].mean()),
        cbc_hgb_mean=('value', lambda x: x[df.loc[x.index, 'cbc_name'] == 'Hgb'].mean()),
        cbc_plt_mean=('value', lambda x: x[df.loc[x.index, 'cbc_name'] == 'Plt'].mean())
    ).reset_index(),
    'FluidBalanceEvents': lambda df: df.groupby('patient_id').agg(
        total_intake=('amount', lambda x: x[df.loc[x.index, 'fluid_type'] == 'Intake'].sum()),
        total_output=('amount', lambda x: x[df.loc[x.index, 'fluid_type'] == 'Output'].sum())
    ).reset_index(),
    'CultureEvents': lambda df: df.groupby('patient_id')['result'].apply(lambda x: ', '.join(x.unique())).reset_index(name='culture_results'),
    'ABGEvents': lambda df: df.groupby('patient_id').agg(
        abg_ph_mean=('value', lambda x: x[df.loc[x.index, 'abg_name'] == 'PH'].mean()),
        abg_pco2_mean=('value', lambda x: x[df.loc[x.index, 'abg_name'] == 'PCO2'].mean()),
        abg_po2_mean=('value', lambda x: x[df.loc[x.index, 'abg_name'] == 'PO2'].mean())
    ).reset_index(),
    'HemoEvents': lambda df: df.groupby('patient_id').agg(
        hemo_heartrate_mean=('value', lambda x: x[df.loc[x.index, 'measurement_name'] == 'HeartRate'].mean()),
        hemo_bpsystolic_mean=('value', lambda x: x[df.loc[x.index, 'measurement_name'] == 'BPSystolic'].mean()),
        hemo_bpdiastolic_mean=('value', lambda x: x[df.loc[x.index, 'measurement_name'] == 'BPDiastolic'].mean())
    ).reset_index(),
    'ChemistryEvents': lambda df: df.groupby('patient_id').agg(
        chem_sodium_mean=('value', lambda x: x[df.loc[x.index, 'chem_name'] == 'Sodium'].mean()),
        chem_potassium_mean=('value', lambda x: x[df.loc[x.index, 'chem_name'] == 'K'].mean()),
        chem_creatinine_mean=('value', lambda x: x[df.loc[x.index, 'chem_name'] == 'Creatinine'].mean())
    ).reset_index()
}

for df_name, df in dataframes.items():
    if df_name in aggregation_strategies:
        print(f"Preparing DataFrame: {df_name}")
        try:
            prepared_df = aggregation_strategies[df_name](df)
            prepared_dataframes[df_name] = prepared_df
            print(f"Successfully prepared {df_name}. Head of prepared data:")
            display(prepared_df.head())
        except Exception as e:
            print(f"Error preparing {df_name}: {e}")
    else:
        print(f"Excluding DataFrame {df_name} from preparation for merging.")

In [ ]:
for df_name, prepared_df in prepared_dataframes.items():
    # Ensure patient_id is the merge key
    if 'patient_id' in prepared_df.columns:
        # Automatically add suffixes to avoid column name conflicts, except for 'patient_id'
        # We assume that only patient_id should not have a suffix
        columns_to_merge = prepared_df.columns.tolist()
        if 'patient_id' in columns_to_merge:
            columns_to_merge.remove('patient_id')

        suffixes = ('', f'_{df_name}') # Suffix for the right dataframe (prepared_df)
        merged_df = pd.merge(merged_df, prepared_df, on='patient_id', how='left', suffixes=suffixes)
        print(f"Successfully merged {df_name} with merged_df.")
        display(merged_df.head())
    else:
        print(f"Warning: DataFrame {df_name} does not have 'patient_id' column and was not merged.")

display(merged_df.info())

#Filtering merge df for adult (>= 18y) with consent for organ donation (OPO table authorized = TRUE)

In [ ]:
merged_df = merged_df[(merged_df['age'] >= 18) & (merged_df['authorized'] == True)].copy()

In [ ]:
merged_df['BD'] = merged_df['brain_death'].astype(int)

# Display the first few rows with the new column
print("DataFrame with new 'BD' column:")
display(merged_df[['brain_death', 'BD']].head())

In [ ]:
# Convert relevant time columns to datetime objects, coercing errors
merged_df['time_referred'] = pd.to_datetime(merged_df['time_referred'], errors='coerce')
merged_df['time_brain_death'] = pd.to_datetime(merged_df['time_brain_death'], errors='coerce')
merged_df['time_asystole'] = pd.to_datetime(merged_df['time_asystole'], errors='coerce')
merged_df['time_procured'] = pd.to_datetime(merged_df['time_procured'], errors='coerce')


# Initialize the 'dtdeath' column
merged_df['dtdeath'] = pd.NA

# Calculate 'dtdeath' based on the 'BD' column
merged_df.loc[merged_df['BD'] == 1, 'dtdeath'] = (merged_df['time_brain_death'] - merged_df['time_referred']).dt.total_seconds() / 3600
merged_df.loc[merged_df['BD'] == 0, 'dtdeath'] = (merged_df['time_asystole'] - merged_df['time_referred']).dt.total_seconds() / 3600

# Display the first few rows with the new 'dtdeath' column
print("DataFrame with new 'dtdeath' column:")
display(merged_df[['time_referred', 'time_brain_death', 'time_asystole', 'BD', 'dtdeath']].head())

In [ ]:
# Calculate 'dt5' based on the 'BD' column
merged_df.loc[merged_df['BD'] == 1, 'dt5'] = (merged_df['time_procured']- merged_df['time_brain_death']).dt.total_seconds() / 3600
merged_df.loc[merged_df['BD'] == 0, 'dt5'] = (merged_df['time_procured']- merged_df['time_asystole']).dt.total_seconds() / 3600

In [ ]:
# Convert relevant time columns to datetime objects, coercing errors
merged_df['time_approached'] = pd.to_datetime(merged_df['time_approached'], errors='coerce')
merged_df['time_authorized'] = pd.to_datetime(merged_df['time_authorized'], errors='coerce')
merged_df['time_procured'] = pd.to_datetime(merged_df['time_procured'], errors='coerce')

# Calculate the new time difference variables in hours
merged_df['dtapp'] = (merged_df['time_approached'] - merged_df['time_referred']).dt.total_seconds() / 3600
merged_df['dtaut'] = (merged_df['time_authorized'] - merged_df['time_referred']).dt.total_seconds() / 3600
merged_df['dtpro'] = (merged_df['time_procured'] - merged_df['time_referred']).dt.total_seconds() / 3600
merged_df['dt4'] = (merged_df['time_authorized'] - merged_df['time_approached']).dt.total_seconds() / 3600

# Display the first few rows with the new columns
print("DataFrame with new time difference columns:")
display(merged_df[['time_referred', 'time_approached', 'time_authorized', 'time_procured', 'dtapp', 'dtaut', 'dtpro', 'dt4']].head())

In [ ]:
df=merged_df.copy()

In [ ]:
df['df5'] = pd.NA
df.loc[df['BD'] == 1, 'df5'] = (df['time_procured'] - df['time_brain_death']).dt.total_seconds() / 3600

print("DataFrame with new 'df5' column for BD = 1:")
display(df[['time_procured', 'time_brain_death', 'BD', 'df5']].head())

In [ ]:
import numpy as np

# Access HemoEvents and ABGEvents DataFrames
hemo_df = dataframes['HemoEvents']
abg_df = dataframes['ABGEvents']

# Convert time columns in HemoEvents to datetime objects
hemo_df['time_event_start'] = pd.to_datetime(hemo_df['time_event_start'], errors='coerce')
hemo_df['time_event_end'] = pd.to_datetime(hemo_df['time_event_end'], errors='coerce')

# Convert time column in ABGEvents to datetime objects
abg_df['time_event'] = pd.to_datetime(abg_df['time_event'], errors='coerce')

# Filter HemoEvents for SBP and DBP
bp_measurements = hemo_df[hemo_df['measurement_name'].isin(['BPSystolic', 'BPDiastolic'])].copy()

# Pivot the filtered DataFrame to get SBP and DBP as columns
pivoted_bp = bp_measurements.pivot_table(
    index=['patient_id', 'time_event_start', 'time_event_end'],
    columns='measurement_name',
    values='value'
).reset_index()

# Rename columns for clarity
pivoted_bp.rename(columns={'BPSystolic': 'SBP', 'BPDiastolic': 'DBP'}, inplace=True)

# Calculate Mean Arterial Pressure (MAP)
pivoted_bp['MAP'] = (pivoted_bp['SBP'] + 2 * pivoted_bp['DBP']) / 3

# Store the modified DataFrames back into the dataframes dictionary
dataframes['HemoEvents_Processed'] = pivoted_bp
dataframes['ABGEvents'] = abg_df # Update ABGEvents in dataframes dict

print("Converted time columns and calculated MAP for HemoEvents.")
print("Head of HemoEvents_Processed with MAP:")
display(dataframes['HemoEvents_Processed'].head())
print("Info for ABGEvents with converted time column:")
display(dataframes['ABGEvents'].info())

In [ ]:
# Create a copy of df to avoid SettingWithCopyWarning
df_temp = df.copy()

# Define the start and end times for each patient based on 'BD' status
df_temp['start_time'] = np.where(
    df_temp['BD'] == 1, df_temp['time_brain_death'], df_temp['time_asystole']
)
df_temp['end_time'] = df_temp['time_procured']

# Select only patient_id and the newly created time window columns
patient_time_windows = df_temp[['patient_id', 'start_time', 'end_time']].dropna(subset=['start_time', 'end_time'])

# Merge time windows with HemoEvents_Processed
hemo_processed_with_windows = pd.merge(
    dataframes['HemoEvents_Processed'],
    patient_time_windows,
    on='patient_id',
    how='inner'
)

# Filter HemoEvents_Processed to only include events within the patient's window
hemo_processed_filtered = hemo_processed_with_windows[
    (hemo_processed_with_windows['time_event_start'] >= hemo_processed_with_windows['start_time']) &
    (hemo_processed_with_windows['time_event_start'] <= hemo_processed_with_windows['end_time'])
].copy()

# Merge time windows with ABGEvents
abg_with_windows = pd.merge(
    dataframes['ABGEvents'],
    patient_time_windows,
    on='patient_id',
    how='inner'
)

# Filter ABGEvents to only include events within the patient's window
abg_filtered = abg_with_windows[
    (abg_with_windows['time_event'] >= abg_with_windows['start_time']) &
    (abg_with_windows['time_event'] <= abg_with_windows['end_time'])
].copy()

# Store the filtered DataFrames back into dataframes for further use
dataframes['HemoEvents_Filtered'] = hemo_processed_filtered
dataframes['ABGEvents_Filtered'] = abg_filtered

print("Defined analysis windows and filtered HemoEvents_Processed and ABGEvents.")
print("Head of HemoEvents_Filtered:")
display(dataframes['HemoEvents_Filtered'].head())
print("Head of ABGEvents_Filtered:")
display(dataframes['ABGEvents_Filtered'].head())

In [ ]:
# Helper function to calculate duration of a condition
def calculate_duration(events_df, condition_col, condition_value, operator, threshold_col, threshold_value):
    if events_df.empty or 'patient_id' not in events_df.columns:
        return pd.DataFrame(columns=['patient_id', f'duration_{condition_col}_{operator}_{threshold_value}'])

    duration_results = []
    for patient_id, group in events_df.groupby('patient_id'):
        # Filter for relevant measurements
        # For SBP and MAP, the values are directly in the respective columns
        # For HR, we need to filter by measurement_name
        if condition_col in ['SBP', 'MAP']:
            relevant_events = group.copy()
        elif condition_col == 'HR':
            hemo_group = dataframes['HemoEvents'][dataframes['HemoEvents']['patient_id'] == patient_id].copy()
            hemo_group['time_event_start'] = pd.to_datetime(hemo_group['time_event_start'], errors='coerce')
            hemo_group['time_event_end'] = pd.to_datetime(hemo_group['time_event_end'], errors='coerce')

            # Filter hemo_group to be within the patient's time window
            patient_window = df_temp[df_temp['patient_id'] == patient_id].iloc[0]
            hemo_group_in_window = hemo_group[
                (hemo_group['time_event_start'] >= patient_window['start_time']) &
                (hemo_group['time_event_start'] <= patient_window['end_time']) &
                (hemo_group['measurement_name'] == 'HeartRate')
            ].copy()
            relevant_events = hemo_group_in_window.rename(columns={'value': 'HR'})
        else:
            relevant_events = pd.DataFrame() # Should not happen with current conditions

        if relevant_events.empty:
            duration_results.append({'patient_id': patient_id, f'duration_{condition_col}_{operator}_{threshold_value}': 0.0})
            continue

        # Apply condition
        if operator == 'lt':
            condition = relevant_events[condition_col] < threshold_value
        elif operator == 'gt':
            condition = relevant_events[condition_col] > threshold_value
        else:
            condition = pd.Series(False, index=relevant_events.index) # Default to no condition if operator is not recognized

        # Calculate time difference between events (assuming 1-hour intervals or event-driven)
        # For simplicity, if we have timestamps, we can calculate the duration each event represents
        # For this dataset, if time_event_start and time_event_end are the same, assume a point measurement
        # and for duration, we might count the number of events * avg_interval, or just count events if intervals are irregular.
        # Given the previous processing, HemoEvents_Processed has start and end times for BP events.
        # For HR, we will assume each record represents a measurement, and we need to consider the time between events.

        # Let's simplify and count events that meet the criteria within the window and then scale by avg interval later
        # or more precisely, consider the 'duration' of the event itself if time_event_start != time_event_end

        # For SBP and MAP, we have start and end times in hemo_processed_filtered
        if condition_col in ['SBP', 'MAP']:
            # Calculate duration for each event where the condition is met
            # If time_event_start and time_event_end are identical, we'll assign a nominal duration (e.g., 1 hour for interval data)
            # Or, we should assume each row represents a measurement at time_event_start and contributes to the state until the next event.
            # Let's count events first for simplicity, then refine if needed.
            count = condition.sum()
            # If each row represents a point measurement and we want duration, we need an interval. Assume 1 hour for now for event count.
            # A more robust approach would be to calculate time between events. Given the data structure, this might be complex without explicit intervals.
            # For now, let's just count the number of events that meet the criteria.

            # More precise duration: If 'time_event_end' is always > 'time_event_start' and represents duration
            # then sum up the durations of the events satisfying the condition.
            # If 'time_event_start' == 'time_event_end', we assume it's a point measurement, let's just count them for now.

            # Let's refine based on the available data, `hemo_processed_filtered` has `time_event_start` and `time_event_end`.
            # If we assume each event's measurement applies for the duration from `time_event_start` to `time_event_end`
            # we can sum up those durations where the condition holds.

            duration_seconds = (relevant_events.loc[condition, 'time_event_end'] - relevant_events.loc[condition, 'time_event_start']).dt.total_seconds().sum()
            total_duration_hours = duration_seconds / 3600.0
            duration_results.append({'patient_id': patient_id, f'duration_{condition_col}_{operator}_{threshold_value}': total_duration_hours})

        elif condition_col == 'HR':
            # For HR, we don't have start/end times per event in the same way. We have original hemo_df with 'time_event_start' and 'time_event_end' for HR.
            # The `hemo_group_in_window` contains the HR values and their `time_event_start` and `time_event_end`.
            duration_seconds = (relevant_events.loc[condition, 'time_event_end'] - relevant_events.loc[condition, 'time_event_start']).dt.total_seconds().sum()
            total_duration_hours = duration_seconds / 3600.0
            duration_results.append({'patient_id': patient_id, f'duration_{condition_col}_{operator}_{threshold_value}': total_duration_hours})

    return pd.DataFrame(duration_results)

# Helper function to calculate AUC over time for deviations (simplified as sum of deviations * nominal interval)
def calculate_auc_deviation(events_df, col_name, threshold, operator, suffix):
    if events_df.empty or 'patient_id' not in events_df.columns:
        return pd.DataFrame(columns=['patient_id', f'auc_{col_name}_{suffix}'])

    auc_results = []
    for patient_id, group in events_df.groupby('patient_id'):
        # For SBP and MAP, values are directly in the group
        # For HR, need to filter original hemo_df for HR values within the window
        if col_name == 'HR':
            hemo_group = dataframes['HemoEvents'][dataframes['HemoEvents']['patient_id'] == patient_id].copy()
            hemo_group['time_event_start'] = pd.to_datetime(hemo_group['time_event_start'], errors='coerce')
            hemo_group['time_event_end'] = pd.to_datetime(hemo_group['time_event_end'], errors='coerce')
            patient_window = df_temp[df_temp['patient_id'] == patient_id].iloc[0]
            relevant_events = hemo_group[
                (hemo_group['time_event_start'] >= patient_window['start_time']) &
                (hemo_group['time_event_start'] <= patient_window['end_time']) &
                (hemo_group['measurement_name'] == 'HeartRate')
            ].copy()
            relevant_events = relevant_events.rename(columns={'value': 'HR'})
            if relevant_events.empty:
                auc_results.append({'patient_id': patient_id, f'auc_{col_name}_{suffix}': 0.0})
                continue
            values = relevant_events['HR']
            event_starts = relevant_events['time_event_start']
            event_ends = relevant_events['time_event_end']
        else:
            values = group[col_name]
            event_starts = group['time_event_start']
            event_ends = group['time_event_end']

        auc = 0.0
        if not values.empty:
            # Calculate AUC as sum of (deviation * duration_of_event_in_hours)
            for i in range(len(values)):
                deviation = 0
                if operator == 'lt' and values.iloc[i] < threshold:
                    deviation = threshold - values.iloc[i]
                elif operator == 'gt' and values.iloc[i] > threshold:
                    deviation = values.iloc[i] - threshold

                if deviation > 0:
                    duration_seconds = (event_ends.iloc[i] - event_starts.iloc[i]).total_seconds()
                    duration_hours = duration_seconds / 3600.0
                    auc += deviation * duration_hours

        auc_results.append({'patient_id': patient_id, f'auc_{col_name}_{suffix}': auc})

    return pd.DataFrame(auc_results)


# Helper function to calculate Coefficient of Variation (CV)
def calculate_cv(events_df, col_name, new_col_name):
    if events_df.empty or 'patient_id' not in events_df.columns:
        return pd.DataFrame(columns=['patient_id', new_col_name])

    cv_results = []
    for patient_id, group in events_df.groupby('patient_id'):
        # For HR, need to filter original hemo_df for HR values within the window
        if col_name == 'HR':
            hemo_group = dataframes['HemoEvents'][dataframes['HemoEvents']['patient_id'] == patient_id].copy()
            hemo_group['time_event_start'] = pd.to_datetime(hemo_group['time_event_start'], errors='coerce')
            hemo_group['time_event_end'] = pd.to_datetime(hemo_group['time_event_end'], errors='coerce')
            patient_window = df_temp[df_temp['patient_id'] == patient_id].iloc[0]
            relevant_events = hemo_group[
                (hemo_group['time_event_start'] >= patient_window['start_time']) &
                (hemo_group['time_event_start'] <= patient_window['end_time']) &
                (hemo_group['measurement_name'] == 'HeartRate')
            ].copy()
            values = relevant_events['value']
        else:
            values = group[col_name]

        if values.mean() != 0:
            cv = values.std() / values.mean()
        else:
            cv = np.nan # Or 0 if standard deviation is also 0
        cv_results.append({'patient_id': patient_id, new_col_name: cv})
    return pd.DataFrame(cv_results)


# Initialize a list to hold all new metrics DataFrames
all_hemo_metrics = []

# 1. Durations of critical physiological states
# SBP < 90 mmHg
sbp_low_duration = calculate_duration(dataframes['HemoEvents_Filtered'], 'SBP', 90, 'lt', 'SBP', 90)
all_hemo_metrics.append(sbp_low_duration)

# MAP < 65 mmHg
map_low_duration = calculate_duration(dataframes['HemoEvents_Filtered'], 'MAP', 65, 'lt', 'MAP', 65)
all_hemo_metrics.append(map_low_duration)

# HR > 120 bpm
hr_high_duration = calculate_duration(dataframes['HemoEvents_Filtered'], 'HR', 120, 'gt', 'HR', 120) # Need to modify calculate_duration to handle HR from original HemoEvents
all_hemo_metrics.append(hr_high_duration)

# HR < 40 bpm
hr_low_duration = calculate_duration(dataframes['HemoEvents_Filtered'], 'HR', 40, 'lt', 'HR', 40)
all_hemo_metrics.append(hr_low_duration)

# 2. AUC for deviations
# SBP < 90 mmHg
sbp_low_auc = calculate_auc_deviation(dataframes['HemoEvents_Filtered'], 'SBP', 90, 'lt', 'low_90')
all_hemo_metrics.append(sbp_low_auc)

# MAP < 65 mmHg
map_low_auc = calculate_auc_deviation(dataframes['HemoEvents_Filtered'], 'MAP', 65, 'lt', 'low_65')
all_hemo_metrics.append(map_low_auc)

# HR > 120 bpm
hr_high_auc = calculate_auc_deviation(dataframes['HemoEvents_Filtered'], 'HR', 120, 'gt', 'high_120')
all_hemo_metrics.append(hr_high_auc)

# HR < 40 bpm
hr_low_auc = calculate_auc_deviation(dataframes['HemoEvents_Filtered'], 'HR', 40, 'lt', 'low_40')
all_hemo_metrics.append(hr_low_auc)

# 3. Coefficient of Variation (CV)
# SBP CV
sbp_cv = calculate_cv(dataframes['HemoEvents_Filtered'], 'SBP', 'cv_sbp')
all_hemo_metrics.append(sbp_cv)

# MAP CV
map_cv = calculate_cv(dataframes['HemoEvents_Filtered'], 'MAP', 'cv_map')
all_hemo_metrics.append(map_cv)

# HR CV
hr_cv = calculate_cv(dataframes['HemoEvents_Filtered'], 'HR', 'cv_hr')
all_hemo_metrics.append(hr_cv)


# Merge all hemo metrics into a single DataFrame
hemo_metrics_df = all_hemo_metrics[0]
for i in range(1, len(all_hemo_metrics)):
    hemo_metrics_df = pd.merge(hemo_metrics_df, all_hemo_metrics[i], on='patient_id', how='outer')

# Merge hemo_metrics_df into the main df
df = pd.merge(df, hemo_metrics_df, on='patient_id', how='left')

print("Calculated critical physiological state metrics (duration, AUC, CV) from HemoEvents and merged them into df.")
print("Head of df with new HemoEvents metrics:")
display(df.head())
print("Info of df with new HemoEvents metrics:")
display(df.info())

In [ ]:
# Helper function to calculate max value for a specific abg_name
def calculate_max_abg_value(abg_df, abg_name, new_col_name):
    if abg_df.empty or 'patient_id' not in abg_df.columns:
        return pd.DataFrame(columns=['patient_id', new_col_name])

    result = abg_df[abg_df['abg_name'] == abg_name].groupby('patient_id')['value'].max().reset_index()
    result.rename(columns={'value': new_col_name}, inplace=True)
    return result

# Helper function to calculate min value for a specific abg_name
def calculate_min_abg_value(abg_df, abg_name, new_col_name):
    if abg_df.empty or 'patient_id' not in abg_df.columns:
        return pd.DataFrame(columns=['patient_id', new_col_name])

    result = abg_df[abg_df['abg_name'] == abg_name].groupby('patient_id')['value'].min().reset_index()
    result.rename(columns={'value': new_col_name}, inplace=True)
    return result

# Helper function to calculate Coefficient of Variation (CV) for a specific abg_name
def calculate_cv_abg(abg_df, abg_name, new_col_name):
    if abg_df.empty or 'patient_id' not in abg_df.columns:
        return pd.DataFrame(columns=['patient_id', new_col_name])

    grouped = abg_df[abg_df['abg_name'] == abg_name].groupby('patient_id')['value']
    std = grouped.std()
    mean = grouped.mean()
    cv = (std / mean).replace([np.inf, -np.inf], np.nan) # Handle division by zero

    result = cv.reset_index(name=new_col_name)
    return result

# Initialize a list to hold all new ABG metrics DataFrames
all_abg_metrics = []

# Calculate maximum 'Lactate'
max_lactate = calculate_max_abg_value(dataframes['ABGEvents_Filtered'], 'Lactate', 'abg_max_lactate')
all_abg_metrics.append(max_lactate)

# Calculate minimum 'HCO3'
min_hco3 = calculate_min_abg_value(dataframes['ABGEvents_Filtered'], 'HCO3', 'abg_min_hco3')
all_abg_metrics.append(min_hco3)

# Calculate CV for 'Lactate'
cv_lactate = calculate_cv_abg(dataframes['ABGEvents_Filtered'], 'Lactate', 'abg_cv_lactate')
all_abg_metrics.append(cv_lactate)

# Calculate CV for 'HCO3'
cv_hco3 = calculate_cv_abg(dataframes['ABGEvents_Filtered'], 'HCO3', 'abg_cv_hco3')
all_abg_metrics.append(cv_hco3)

# Merge all ABG metrics into a single DataFrame
abg_metrics_df = all_abg_metrics[0]
for i in range(1, len(all_abg_metrics)):
    abg_metrics_df = pd.merge(abg_metrics_df, all_abg_metrics[i], on='patient_id', how='outer')

# Merge abg_metrics_df into the main df
df = pd.merge(df, abg_metrics_df, on='patient_id', how='left')

print("Calculated ABG metrics (max lactate, min HCO3, CV for lactate and HCO3) and merged them into df.")
print("Head of df with new ABG metrics:")
display(df.head())
print("Info of df with new ABG metrics:")
display(df.info())

### Summary of Newly Added Variables

**From HemoEvents (Hemodynamic Metrics):**

*   **`duration_SBP_lt_90`**: Duration (in hours) where Systolic Blood Pressure (SBP) was less than 90 mmHg.
*   **`duration_MAP_lt_65`**: Duration (in hours) where Mean Arterial Pressure (MAP) was less than 65 mmHg.
*   **`duration_HR_gt_120`**: Duration (in hours) where Heart Rate (HR) was greater than 120 bpm.
*   **`duration_HR_lt_40`**: Duration (in hours) where Heart Rate (HR) was less than 40 bpm.
*   **`auc_SBP_low_90`**: Area Under the Curve for SBP deviations below 90 mmHg.
*   **`auc_MAP_low_65`**: Area Under the Curve for MAP deviations below 65 mmHg.
*   **`auc_HR_high_120`**: Area Under the Curve for HR deviations above 120 bpm.
*   **`auc_HR_low_40`**: Area Under the Curve for HR deviations below 40 bpm.
*   **`cv_sbp`**: Coefficient of Variation for SBP.
*   **`cv_map`**: Coefficient of Variation for MAP.
*   **`cv_hr`**: Coefficient of Variation for Heart Rate.

**From ABGEvents (Arterial Blood Gas Metrics):**

*   **`abg_max_lactate`**: Maximum Lactate value observed. (Note: This column is **null** as 'Lactate' was not found in the `abg_name` column.)
*   **`abg_min_hco3`**: Minimum HCO3 value observed.
*   **`abg_cv_lactate`**: Coefficient of Variation for Lactate. (Note: This column is **null** as 'Lactate' was not found in the `abg_name` column.)
*   **`abg_cv_hco3`**: Coefficient of Variation for HCO3.


In [ ]:
df

In [ ]:
df.to_csv('new_ORCHID.csv', index=False)
print("DataFrame 'df' exported to 'new_ORCHID.csv'")

In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# TABLE I – Baseline characteristics of the study population
# =========================================================

# usa a tua base final
table_df = df.copy()

# ---------- opcional: restringir exatamente à coorte do paper ----------
# Se quiseres garantir apenas adultos + autorizados:
# table_df = table_df[(table_df["age"] >= 18) & (table_df["authorized"] == True)].copy()

# ---------- definir variáveis ----------
continuous_vars = [
    "age",
    "height_in",
    "weight_kg",
    "chem_creatinine_mean",
    "chem_sodium_mean",
    "chem_potassium_mean",
    "cbc_wbc_mean",
    "cbc_hgb_mean",
    "cbc_plt_mean",
    "abg_ph_mean",
    "abg_pco2_mean",
    "abg_po2_mean",
    "abg_min_hco3",
    "hemo_heartrate_mean",
    "hemo_bpsystolic_mean",
    "hemo_bpdiastolic_mean",
    "duration_MAP_lt_65",
    "duration_SBP_lt_90",
    "duration_HR_gt_120",
    "duration_HR_lt_40",
    "auc_MAP_low_65",
    "auc_SBP_low_90",
    "auc_HR_high_120",
    "auc_HR_low_40",
    "cv_map",
    "cv_sbp",
    "cv_hr",
    "total_intake",
    "total_output",
    "dtdeath",
    "dt5",
    "dtapp",
    "dtaut",
    "dtpro",
]

categorical_vars = [
    "gender",
    "race",
    "brain_death",
    "authorized",
    "procured",
    "transplanted",
    "approached",
    "tissue_referral",
    "eye_referral",
    "abo_blood_type",
    "abo_rh",
    "referral_day_of_week",
]

# manter só as que existem
continuous_vars = [v for v in continuous_vars if v in table_df.columns]
categorical_vars = [v for v in categorical_vars if v in table_df.columns]

# ---------- escolher formato por variável contínua ----------
# Se quiseres forçar algumas como mediana (IQR), coloca aqui:
force_median_iqr = {
    "total_intake",
    "total_output",
    "duration_MAP_lt_65",
    "duration_SBP_lt_90",
    "duration_HR_gt_120",
    "duration_HR_lt_40",
    "auc_MAP_low_65",
    "auc_SBP_low_90",
    "auc_HR_high_120",
    "auc_HR_low_40",
    "dtdeath",
    "dt5",
    "dtapp",
    "dtaut",
    "dtpro",
}

def fmt_mean_sd(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    return f"{x.mean():.2f} ({x.std():.2f})"

def fmt_median_iqr(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return "NA"
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    return f"{x.median():.2f} ({q1:.2f}–{q3:.2f})"

def is_skewed(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) < 20:
        return False
    return abs(x.skew()) > 1

rows = []

# ---------- N total ----------
rows.append({
    "Characteristic": "Total study population",
    "Overall": f"{len(table_df):,}"
})

# ---------- contínuas ----------
for var in continuous_vars:
    series = table_df[var]

    if var in force_median_iqr or is_skewed(series):
        value = fmt_median_iqr(series)
    else:
        value = fmt_mean_sd(series)

    rows.append({
        "Characteristic": var,
        "Overall": value
    })

# ---------- categóricas ----------
for var in categorical_vars:
    vc = table_df[var].fillna("Missing").value_counts(dropna=False)
    total = len(table_df)

    first = True
    for level, count in vc.items():
        pct = 100 * count / total
        rows.append({
            "Characteristic": f"{var}" if first else f"  {level}",
            "Overall": "" if first else f"{count} ({pct:.1f}%)"
        })
        first = False

table1 = pd.DataFrame(rows)

# ---------- nomes bonitos ----------
pretty_names = {
    "age": "Age, years",
    "height_in": "Height, inches",
    "weight_kg": "Weight, kg",
    "gender": "Sex",
    "race": "Race",
    "brain_death": "Brain death status",
    "authorized": "Authorized for donation",
    "procured": "Procured",
    "transplanted": "Transplanted",
    "approached": "Approached",
    "tissue_referral": "Tissue referral",
    "eye_referral": "Eye referral",
    "abo_blood_type": "ABO blood type",
    "abo_rh": "Rh factor",
    "referral_day_of_week": "Referral day of week",
    "chem_creatinine_mean": "Creatinine",
    "chem_sodium_mean": "Sodium",
    "chem_potassium_mean": "Potassium",
    "cbc_wbc_mean": "White blood cell count",
    "cbc_hgb_mean": "Hemoglobin",
    "cbc_plt_mean": "Platelets",
    "abg_ph_mean": "pH",
    "abg_pco2_mean": "pCO2",
    "abg_po2_mean": "pO2",
    "abg_min_hco3": "HCO3 (minimum)",
    "hemo_heartrate_mean": "Heart rate",
    "hemo_bpsystolic_mean": "Systolic blood pressure",
    "hemo_bpdiastolic_mean": "Diastolic blood pressure",
    "duration_MAP_lt_65": "Duration MAP <65",
    "duration_SBP_lt_90": "Duration SBP <90",
    "duration_HR_gt_120": "Duration HR >120",
    "duration_HR_lt_40": "Duration HR <40",
    "auc_MAP_low_65": "AUC MAP <65",
    "auc_SBP_low_90": "AUC SBP <90",
    "auc_HR_high_120": "AUC HR >120",
    "auc_HR_low_40": "AUC HR <40",
    "cv_map": "MAP coefficient of variation",
    "cv_sbp": "SBP coefficient of variation",
    "cv_hr": "HR coefficient of variation",
    "total_intake": "Total intake",
    "total_output": "Total output",
    "dtdeath": "Time to death event, h",
    "dt5": "Time from death event to procurement, h",
    "dtapp": "Time to approach, h",
    "dtaut": "Time to authorization, h",
    "dtpro": "Time to procurement, h",
}

table1["Characteristic"] = table1["Characteristic"].replace(pretty_names)

# ---------- mostrar ----------
display(table1)

# ---------- exportar ----------
table1.to_excel("Table_I_baseline_characteristics.xlsx", index=False)
table1.to_csv("Table_I_baseline_characteristics.csv", index=False)

print("Saved: Table_I_baseline_characteristics.xlsx")
print("Saved: Table_I_baseline_characteristics.csv")

In [ ]:
import pandas as pd
df = pd.read_csv('path_to_orchid_data.csv')

##PIPELINE SCENARIO 1
Brain-dead donors who proceeded to procurement.
Outcome: transplanted versus not transplanted.

In [ ]:
# ============================================================
# SCENARIO 1 PIPELINE — FEATURE SELECTION + CLUSTERING
# ============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

import statsmodels.api as sm
import umap.umap_ as umap

# =====================
# 1. SCENARIO 1 FILTER
# =====================
df_s1 = df[
    (df["brain_death"] == True) &
    (df["procured"] == 1)
].copy()

print("N =", len(df_s1))

# =====================
# 2. DEFINE OUTCOME
# =====================
outcome = "transplanted"
df_s1[outcome] = df_s1[outcome].astype(int)

# =====================
# 3. RAW FEATURE POOL
# =====================
candidate_features = [
    # Demographics
    "age", "weight_kg", "height_in",

    # Labs
    "chem_creatinine_mean", "chem_sodium_mean", "chem_potassium_mean",
    "cbc_wbc_mean", "cbc_hgb_mean", "cbc_plt_mean",

    # Gasometry
    "abg_ph_mean", "abg_pco2_mean", "abg_po2_mean",
    "abg_min_hco3",

    # Hemodynamics
    "hemo_heartrate_mean", "hemo_bpsystolic_mean", "hemo_bpdiastolic_mean",

    # Instability / burden
    "duration_MAP_lt_65", "duration_SBP_lt_90",
    "duration_HR_gt_120", "duration_HR_lt_40",
    "auc_MAP_low_65", "auc_SBP_low_90",
    "auc_HR_high_120", "auc_HR_low_40",

    # Variability
    "cv_map", "cv_sbp", "cv_hr",

    # Fluids
    "total_intake", "total_output",

    # Timing (optional but interesting)
    "dtdeath", "dt5", "dtapp", "dtaut", "dtpro"
]

features = [f for f in candidate_features if f in df_s1.columns]

# =====================
# 4. MISSINGNESS FILTER
# =====================
missing = df_s1[features].isna().mean()

features = missing[missing < 0.4].index.tolist()

print("\nFeatures after missing filter:")
print(features)

# =====================
# 5. IMPUTATION
# =====================
X = df_s1[features]

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=features)

# =====================
# 6. REMOVE LOW VARIANCE
# =====================
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X_imp)

features_var = np.array(features)[selector.get_support()]

X_var = pd.DataFrame(X_var, columns=features_var)

print("\nFeatures after variance filter:")
print(features_var)

# =====================
# 7. REMOVE HIGHLY CORRELATED FEATURES
# =====================
corr = X_var.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

X_uncorr = X_var.drop(columns=to_drop)

print("\nDropped due to high correlation:")
print(to_drop)

print("\nFinal features for clustering:")
print(X_uncorr.columns.tolist())

# =====================
# 8. STANDARDIZATION
# =====================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_uncorr)

# =====================
# 9. K SELECTION
# =====================
sil_scores = []

for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    sil_scores.append((k, sil))

print("\nSilhouette scores:", sil_scores)

k_opt = max(sil_scores, key=lambda x: x[1])[0]

# =====================
# 10. FINAL CLUSTERING
# =====================
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=20)
df_s1["cluster"] = kmeans.fit_predict(X_scaled)

print("\nCluster distribution:")
print(df_s1["cluster"].value_counts())

# =====================
# 11. GMM SENSITIVITY
# =====================
gmm = GaussianMixture(n_components=k_opt, random_state=42)
df_s1["cluster_gmm"] = gmm.fit_predict(X_scaled)

print("\nKMeans vs GMM:")
print(pd.crosstab(df_s1["cluster"], df_s1["cluster_gmm"]))

# =====================
# 12. CLUSTER CHARACTERIZATION
# =====================
cluster_means = df_s1.groupby("cluster")[X_uncorr.columns].mean()
print("\nCluster means:")
print(cluster_means)

# =====================
# 13. FEATURE IMPORTANCE (CLUSTER DEFINITION)
# =====================
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_scaled, df_s1["cluster"])

importance = pd.Series(rf.feature_importances_, index=X_uncorr.columns)
importance = importance.sort_values(ascending=False)

print("\nTop variables defining clusters:")
print(importance.head(10))

# =====================
# 14. OUTCOME BY CLUSTER
# =====================
print("\nTransplant rate:")
print(df_s1.groupby("cluster")[outcome].mean())

# =====================
# 15. LOGISTIC MODEL
# =====================
X_model = sm.add_constant(df_s1[["cluster", "age"]])
y = df_s1[outcome]

model = sm.Logit(y, X_model).fit()
print(model.summary())

# =====================
# 16. UMAP
# =====================
reducer = umap.UMAP(random_state=42)
emb = reducer.fit_transform(X_scaled)

df_s1["umap1"] = emb[:,0]
df_s1["umap2"] = emb[:,1]

# =====================
# 17. PLOTS
# =====================

# UMAP
sns.scatterplot(data=df_s1, x="umap1", y="umap2", hue="cluster")
plt.title("Clusters")
plt.show()

# Outcome
sns.scatterplot(data=df_s1, x="umap1", y="umap2", hue=outcome)
plt.title("Outcome")
plt.show()

# Barplot
sns.barplot(data=df_s1, x="cluster", y=outcome)
plt.title("Transplant rate")
plt.show()

# Feature importance
sns.barplot(x=importance.values[:10], y=importance.index[:10])
plt.title("Top features driving clusters")
plt.show()

#OPO influence in TX outcome in Scenario 1

In [ ]:
# ============================================================
# SCENARIO 1 — OPO EFFECT AFTER PHYSIOLOGICAL CLUSTERING
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# -----------------------------
# 0. COPY + BASIC PREPARATION
# -----------------------------
df_opo = df_s1.copy()

outcome = "transplanted"
df_opo[outcome] = df_opo[outcome].astype(int)

df_opo["cluster"] = df_opo["cluster"].astype("category")
df_opo["opo"] = df_opo["opo"].astype("category")

df_opo = df_opo.dropna(subset=["opo", "cluster", "age", outcome]).copy()

print("N for OPO analysis =", len(df_opo))
print("Number of clusters =", df_opo["cluster"].nunique())
print("Number of OPOs =", df_opo["opo"].nunique())

# -----------------------------
# 1. DESCRIPTIVE TABLES
# -----------------------------
print("\nTRANSPLANT RATE BY CLUSTER")
cluster_rates = df_opo.groupby("cluster")[outcome].agg(["mean", "sum", "count"])
cluster_rates["percent"] = cluster_rates["mean"] * 100
print(cluster_rates)

print("\nTRANSPLANT RATE BY OPO")
opo_rates = df_opo.groupby("opo")[outcome].agg(["mean", "sum", "count"]).sort_values("mean", ascending=False)
opo_rates["percent"] = opo_rates["mean"] * 100
print(opo_rates)

print("\nTRANSPLANT RATE BY CLUSTER × OPO")
cluster_opo_table = df_opo.pivot_table(
    index="cluster",
    columns="opo",
    values=outcome,
    aggfunc=["mean", "count"]
)
print(cluster_opo_table)

# -----------------------------
# 2. HEATMAP
# -----------------------------
pivot_rate = df_opo.pivot_table(
    index="cluster",
    columns="opo",
    values=outcome,
    aggfunc="mean"
)

plt.figure(figsize=(max(10, pivot_rate.shape[1] * 0.6), 5))
sns.heatmap(pivot_rate, annot=True, fmt=".2f", cmap="gray")
plt.title("Transplant rate by physiological cluster and OPO")
plt.tight_layout()
plt.show()

# -----------------------------
# 3. FILTER SMALL OPOs
# -----------------------------
min_n_opo = 10
valid_opos = df_opo["opo"].value_counts()
valid_opos = valid_opos[valid_opos >= min_n_opo].index

df_opo_filt = df_opo[df_opo["opo"].isin(valid_opos)].copy()
df_opo_filt["opo"] = df_opo_filt["opo"].astype("category")
df_opo_filt["cluster"] = df_opo_filt["cluster"].astype("category")

print("\nAfter filtering OPOs >=", min_n_opo)
print("N =", len(df_opo_filt))
print("OPOs kept =", df_opo_filt['opo'].nunique())

# -----------------------------
# 4. LOGISTIC MODEL
# -----------------------------
print("\nMODEL 1: cluster + opo + age")

formula_main = f"{outcome} ~ C(cluster) + C(opo) + age"
model_main = smf.logit(formula=formula_main, data=df_opo_filt).fit(disp=False)
print(model_main.summary())

or_main = pd.DataFrame({
    "OR": np.exp(model_main.params),
    "CI_lower": np.exp(model_main.conf_int()[0]),
    "CI_upper": np.exp(model_main.conf_int()[1]),
    "p_value": model_main.pvalues
}).sort_values("p_value")

print("\nOdds ratios:")
print(or_main)

# -----------------------------
# 5. INTERACTION MODEL
# -----------------------------
print("\nMODEL 2: cluster * opo")

formula_int = f"{outcome} ~ C(cluster) * C(opo) + age"

try:
    model_int = smf.logit(formula=formula_int, data=df_opo_filt).fit(disp=False, maxiter=200)
    print(model_int.summary())

    lr_stat = 2 * (model_int.llf - model_main.llf)
    df_diff = model_int.df_model - model_main.df_model
    lr_p = chi2.sf(lr_stat, df_diff)

    print("\nInteraction test:")
    print(f"LR = {lr_stat:.3f} | p = {lr_p:.5f}")

except Exception as e:
    print("Interaction model failed:", e)

# -----------------------------
# 6. WITHIN-CLUSTER ANALYSIS
# -----------------------------
print("\nWITHIN-CLUSTER OPO EFFECT")

results = []

for cl in df_opo_filt["cluster"].cat.categories:
    sub = df_opo_filt[df_opo_filt["cluster"] == cl].copy()

    if len(sub) < 20:
        continue

    valid_sub_opos = sub["opo"].value_counts()
    valid_sub_opos = valid_sub_opos[valid_sub_opos >= 5].index
    sub = sub[sub["opo"].isin(valid_sub_opos)]

    if sub["opo"].nunique() < 2:
        continue

    print(f"\nCluster {cl}")
    print(sub.groupby("opo")[outcome].mean().sort_values(ascending=False))

    try:
        m = smf.logit(f"{outcome} ~ C(opo) + age", data=sub).fit(disp=False)
        null = smf.logit(f"{outcome} ~ age", data=sub).fit(disp=False)

        lr = 2 * (m.llf - null.llf)
        p = chi2.sf(lr, m.df_model - null.df_model)

        results.append({"cluster": cl, "p_value": p})

        print("OPO effect p =", p)

    except:
        pass

results = pd.DataFrame(results)
print("\nSummary:")
print(results)

# -----------------------------
# 7. PREDICTED PROBABILITIES
# -----------------------------
mean_age = df_opo_filt["age"].mean()

grid = pd.DataFrame([
    {"cluster": cl, "opo": opo, "age": mean_age}
    for cl in df_opo_filt["cluster"].cat.categories
    for opo in df_opo_filt["opo"].cat.categories
])

grid["pred"] = model_main.predict(grid)

pivot_pred = grid.pivot(index="cluster", columns="opo", values="pred")

plt.figure(figsize=(max(10, pivot_pred.shape[1] * 0.6), 5))
sns.heatmap(pivot_pred, annot=True, fmt=".2f", cmap="gray")
plt.title("Predicted probability (adjusted)")
plt.tight_layout()
plt.show()

# -----------------------------
# 8. BARPLOT
# -----------------------------
plt.figure(figsize=(12, 6))
sns.barplot(data=df_opo_filt, x="cluster", y=outcome, hue="opo", errorbar=None)
plt.title("Observed transplant rate")
plt.tight_layout()
plt.show()

# -----------------------------
# 9. SAVE
# -----------------------------
cluster_rates.to_excel("cluster_rates.xlsx")
opo_rates.to_excel("opo_rates.xlsx")
cluster_opo_table.to_excel("cluster_opo.xlsx")
pivot_pred.to_excel("predicted_cluster_opo.xlsx")
results.to_excel("within_cluster_opo.xlsx")

print("\nFiles saved.")

# PIPELINE SCENARIO 2

Non Brain-dead donors who proceeded to procurement.
Outcome: transplanted versus not transplanted.

In [ ]:
# ============================================================
# SCENARIO 2 PIPELINE — NON-BRAIN-DEATH (DCD-like)
# ============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

import statsmodels.api as sm
import umap.umap_ as umap

# =====================
# 1. SCENARIO FILTER
# =====================
df_s2 = df[
    (df["brain_death"] == False) &
    (df["procured"] == 1)
].copy()

print("Scenario 2 N =", len(df_s2))

# =====================
# 2. OUTCOME
# =====================
outcome = "transplanted"
df_s2[outcome] = df_s2[outcome].astype(int)

# =====================
# 3. FEATURES (igual ao S1)
# =====================
candidate_features = [
    "age", "weight_kg", "height_in",
    "chem_creatinine_mean", "chem_sodium_mean", "chem_potassium_mean",
    "cbc_wbc_mean", "cbc_hgb_mean",
    "abg_ph_mean", "abg_pco2_mean", "abg_po2_mean", "abg_min_hco3",
    "hemo_heartrate_mean", "hemo_bpsystolic_mean", "hemo_bpdiastolic_mean",
    "total_intake", "total_output"
]

features = [f for f in candidate_features if f in df_s2.columns]

# =====================
# 4. MISSING FILTER
# =====================
missing = df_s2[features].isna().mean()
features = missing[missing < 0.4].index.tolist()

print("\nFeatures after missing filter:")
print(features)

# =====================
# 5. IMPUTATION
# =====================
X = df_s2[features]

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=features)

# =====================
# 6. VARIANCE FILTER
# =====================
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X_imp)

features_var = np.array(features)[selector.get_support()]
X_var = pd.DataFrame(X_var, columns=features_var)

# =====================
# 7. CORRELATION FILTER
# =====================
corr = X_var.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
X_uncorr = X_var.drop(columns=to_drop)

print("\nFinal features:")
print(X_uncorr.columns.tolist())

# =====================
# 8. STANDARDIZE
# =====================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_uncorr)

# =====================
# 9. CLUSTER SELECTION
# =====================
sil_scores = []

for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    sil_scores.append((k, silhouette_score(X_scaled, labels)))

print("\nSilhouette:", sil_scores)

k_opt = max(sil_scores, key=lambda x: x[1])[0]

# =====================
# 10. FINAL CLUSTERING
# =====================
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=20)
df_s2["cluster"] = kmeans.fit_predict(X_scaled)

print("\nCluster distribution:")
print(df_s2["cluster"].value_counts())

# =====================
# 11. GMM SENSITIVITY
# =====================
gmm = GaussianMixture(n_components=k_opt, random_state=42)
df_s2["cluster_gmm"] = gmm.fit_predict(X_scaled)

print("\nKMeans vs GMM:")
print(pd.crosstab(df_s2["cluster"], df_s2["cluster_gmm"]))

# =====================
# 12. CLUSTER DESCRIPTION
# =====================
cluster_means = df_s2.groupby("cluster")[X_uncorr.columns].mean()
print("\nCluster means:")
print(cluster_means)

# =====================
# 13. FEATURE IMPORTANCE
# =====================
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_scaled, df_s2["cluster"])

importance = pd.Series(rf.feature_importances_, index=X_uncorr.columns)
importance = importance.sort_values(ascending=False)

print("\nTop features:")
print(importance.head(10))

# =====================
# 14. OUTCOME
# =====================
print("\nTransplant rate:")
print(df_s2.groupby("cluster")[outcome].mean())

# =====================
# 15. LOGISTIC MODEL
# =====================
X_model = sm.add_constant(df_s2[["cluster", "age"]])
y = df_s2[outcome]

model = sm.Logit(y, X_model).fit()
print(model.summary())

# =====================
# 16. UMAP
# =====================
reducer = umap.UMAP(random_state=42)
emb = reducer.fit_transform(X_scaled)

df_s2["umap1"] = emb[:,0]
df_s2["umap2"] = emb[:,1]

# =====================
# 17. PLOTS
# =====================

# Clusters
sns.scatterplot(data=df_s2, x="umap1", y="umap2", hue="cluster")
plt.title("Scenario 2 Clusters")
plt.show()

# Outcome
sns.scatterplot(data=df_s2, x="umap1", y="umap2", hue=outcome)
plt.title("Outcome")
plt.show()

# Transplant rate
sns.barplot(data=df_s2, x="cluster", y=outcome)
plt.title("Transplant rate")
plt.show()

# Feature importance
sns.barplot(x=importance.values[:10], y=importance.index[:10])
plt.title("Top drivers")
plt.show()

In [ ]:
# ============================================================
# SCENARIO 2 — OPO EFFECT AFTER PHYSIOLOGICAL CLUSTERING
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# -----------------------------
# 0. COPY + BASIC PREPARATION
# -----------------------------
df_opo = df_s2.copy()

outcome = "transplanted"
df_opo[outcome] = df_opo[outcome].astype(int)

df_opo["cluster"] = df_opo["cluster"].astype("category")
df_opo["opo"] = df_opo["opo"].astype("category")

df_opo = df_opo.dropna(subset=["opo", "cluster", "age", outcome]).copy()

print("N for OPO analysis =", len(df_opo))
print("Number of clusters =", df_opo["cluster"].nunique())
print("Number of OPOs =", df_opo["opo"].nunique())

# -----------------------------
# 1. DESCRIPTIVE TABLES
# -----------------------------
print("\nTRANSPLANT RATE BY CLUSTER")
cluster_rates = df_opo.groupby("cluster", observed=False)[outcome].agg(["mean", "sum", "count"])
cluster_rates["percent"] = cluster_rates["mean"] * 100
print(cluster_rates)

print("\nTRANSPLANT RATE BY OPO")
opo_rates = df_opo.groupby("opo", observed=False)[outcome].agg(["mean", "sum", "count"]).sort_values("mean", ascending=False)
opo_rates["percent"] = opo_rates["mean"] * 100
print(opo_rates)

print("\nTRANSPLANT RATE BY CLUSTER × OPO")
cluster_opo_table = df_opo.pivot_table(
    index="cluster",
    columns="opo",
    values=outcome,
    aggfunc=["mean", "count"],
    observed=False
)
print(cluster_opo_table)

# -----------------------------
# 2. HEATMAP
# -----------------------------
pivot_rate = df_opo.pivot_table(
    index="cluster",
    columns="opo",
    values=outcome,
    aggfunc="mean",
    observed=False
)

plt.figure(figsize=(max(10, pivot_rate.shape[1] * 0.6), 5))
sns.heatmap(pivot_rate, annot=True, fmt=".2f", cmap="gray")
plt.title("Transplant rate by physiological cluster and OPO")
plt.tight_layout()
plt.show()

# -----------------------------
# 3. FILTER SMALL OPOs
# -----------------------------
min_n_opo = 10
valid_opos = df_opo["opo"].value_counts()
valid_opos = valid_opos[valid_opos >= min_n_opo].index

df_opo_filt = df_opo[df_opo["opo"].isin(valid_opos)].copy()
df_opo_filt["opo"] = df_opo_filt["opo"].astype("category")
df_opo_filt["cluster"] = df_opo_filt["cluster"].astype("category")

print("\nAfter filtering OPOs >=", min_n_opo)
print("N =", len(df_opo_filt))
print("OPOs kept =", df_opo_filt["opo"].nunique())

# -----------------------------
# 4. LOGISTIC MODEL
# -----------------------------
print("\nMODEL 1: cluster + opo + age")

formula_main = f"{outcome} ~ C(cluster) + C(opo) + age"
model_main = smf.logit(formula=formula_main, data=df_opo_filt).fit(disp=False)
print(model_main.summary())

or_main = pd.DataFrame({
    "OR": np.exp(model_main.params),
    "CI_lower": np.exp(model_main.conf_int()[0]),
    "CI_upper": np.exp(model_main.conf_int()[1]),
    "p_value": model_main.pvalues
}).sort_values("p_value")

print("\nOdds ratios:")
print(or_main)

# -----------------------------
# 5. INTERACTION MODEL
# -----------------------------
print("\nMODEL 2: cluster * opo")

formula_int = f"{outcome} ~ C(cluster) * C(opo) + age"

try:
    model_int = smf.logit(formula=formula_int, data=df_opo_filt).fit(disp=False, maxiter=200)
    print(model_int.summary())

    lr_stat = 2 * (model_int.llf - model_main.llf)
    df_diff = model_int.df_model - model_main.df_model
    lr_p = chi2.sf(lr_stat, df_diff)

    print("\nInteraction test:")
    print(f"LR = {lr_stat:.3f} | p = {lr_p:.5f}")

except Exception as e:
    print("Interaction model failed:", e)

# -----------------------------
# 6. WITHIN-CLUSTER ANALYSIS
# -----------------------------
print("\nWITHIN-CLUSTER OPO EFFECT")

results = []

for cl in df_opo_filt["cluster"].cat.categories:
    sub = df_opo_filt[df_opo_filt["cluster"] == cl].copy()

    if len(sub) < 20:
        continue

    valid_sub_opos = sub["opo"].value_counts()
    valid_sub_opos = valid_sub_opos[valid_sub_opos >= 5].index
    sub = sub[sub["opo"].isin(valid_sub_opos)].copy()

    if sub["opo"].nunique() < 2:
        continue

    print(f"\nCluster {cl}")
    print(sub.groupby("opo", observed=False)[outcome].mean().sort_values(ascending=False))

    try:
        m = smf.logit(f"{outcome} ~ C(opo) + age", data=sub).fit(disp=False)
        null = smf.logit(f"{outcome} ~ age", data=sub).fit(disp=False)

        lr = 2 * (m.llf - null.llf)
        p = chi2.sf(lr, m.df_model - null.df_model)

        results.append({"cluster": cl, "p_value": p})

        print("OPO effect p =", p)

    except:
        pass

results = pd.DataFrame(results)
print("\nSummary:")
print(results)

# -----------------------------
# 7. PREDICTED PROBABILITIES
# -----------------------------
mean_age = df_opo_filt["age"].mean()

grid = pd.DataFrame([
    {"cluster": cl, "opo": opo, "age": mean_age}
    for cl in df_opo_filt["cluster"].cat.categories
    for opo in df_opo_filt["opo"].cat.categories
])

grid["pred"] = model_main.predict(grid)

pivot_pred = grid.pivot(index="cluster", columns="opo", values="pred")

plt.figure(figsize=(max(10, pivot_pred.shape[1] * 0.6), 5))
sns.heatmap(pivot_pred, annot=True, fmt=".2f", cmap="gray")
plt.title("Predicted probability (adjusted)")
plt.tight_layout()
plt.show()

# -----------------------------
# 8. BARPLOT
# -----------------------------
plt.figure(figsize=(12, 6))
sns.barplot(data=df_opo_filt, x="cluster", y=outcome, hue="opo", errorbar=None)
plt.title("Observed transplant rate")
plt.tight_layout()
plt.show()

# -----------------------------
# 9. SAVE
# -----------------------------
cluster_rates.to_excel("scenario2_cluster_rates.xlsx")
opo_rates.to_excel("scenario2_opo_rates.xlsx")
cluster_opo_table.to_excel("scenario2_cluster_opo.xlsx")
pivot_pred.to_excel("scenario2_predicted_cluster_opo.xlsx")
results.to_excel("scenario2_within_cluster_opo.xlsx")

print("\nFiles saved.")

# PIPELINE SCENARIO 3

Brain-dead donors.
Outcome: Procured versus not procured.
Variaveis fisiológicas e variáveis processo.

In [ ]:
# ============================================================
# SCENARIO 3 — BD donors, outcome = procurement (FULL PIPELINE)
# ============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

import statsmodels.api as sm
import umap.umap_ as umap

# =====================
# 1. FILTER
# =====================
df_s3 = df[df["brain_death"] == True].copy()

print("Scenario 3 N =", len(df_s3))

# =====================
# 2. OUTCOME
# =====================
outcome = "procured"
df_s3[outcome] = df_s3[outcome].astype(int)

# =====================
# 3. FEATURES — PHYSIOLOGY ONLY
# =====================
candidate_features = [
    "age", "weight_kg", "height_in",
    "chem_creatinine_mean", "chem_sodium_mean", "chem_potassium_mean",
    "cbc_wbc_mean", "cbc_hgb_mean",
    "abg_ph_mean", "abg_pco2_mean", "abg_po2_mean",
    "hemo_heartrate_mean", "hemo_bpsystolic_mean", "hemo_bpdiastolic_mean",
    "total_intake", "total_output"
]

features = [f for f in candidate_features if f in df_s3.columns]

# =====================
# 4. MISSING FILTER
# =====================
missing = df_s3[features].isna().mean()
features = missing[missing < 0.4].index.tolist()

print("\nFeatures after missing filter:")
print(features)

# =====================
# 5. IMPUTATION
# =====================
X = df_s3[features].copy() # Ensure X has the original index

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=features, index=X.index)

# =====================
# 6. VARIANCE FILTER
# =====================
selector = VarianceThreshold(threshold=0.01)
X_var_array = selector.fit_transform(X_imp)

features_var = np.array(features)[selector.get_support()]
X_var = pd.DataFrame(X_var_array, columns=features_var, index=X_imp.index) # Preserve index

# =====================
# 7. CORRELATION FILTER
# =====================
corr = X_var.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
X_final = X_var.drop(columns=to_drop) # This will naturally preserve the index

print("\nFinal features:")
print(X_final.columns.tolist())

# =====================
# 8. STANDARDIZE
# =====================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# =====================
# 9. CLUSTER SELECTION
# =====================
sil_scores = []

for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    sil_scores.append((k, silhouette_score(X_scaled, labels)))

print("\nSilhouette scores:", sil_scores)

# Forçar interpretabilidade
k_opt = 2

# =====================
# 10. FINAL CLUSTERING
# =====================
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=20)
df_s3["cluster"] = kmeans.fit_predict(X_scaled)

print("\nCluster distribution:")
print(df_s3["cluster"].value_counts())

# =====================
# 11. CLUSTER DESCRIPTION
# =====================
cluster_means = df_s3.groupby("cluster")[X_final.columns].mean()

print("\nCluster means:")
print(cluster_means)

# =====================
# 12. FEATURE IMPORTANCE
# =====================
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_scaled, df_s3["cluster"])

importance = pd.Series(rf.feature_importances_, index=X_final.columns)
importance = importance.sort_values(ascending=False)

print("\nTop variables defining clusters:")
print(importance.head(10))

# =====================
# 13. OUTCOME BY CLUSTER
# =====================
proc_rates = df_s3.groupby("cluster")[outcome].mean()

print("\nProcurement rate by cluster:")
print(proc_rates)

# =====================
# 14. LOGISTIC REGRESSION
# =====================
X_model = sm.add_constant(df_s3[["cluster", "age"]])
y = df_s3[outcome]

model = sm.Logit(y, X_model).fit()

print("\nLogistic regression:")
print(model.summary())

# =====================
# 15. UMAP
# =====================
reducer = umap.UMAP(random_state=42)
embedding = reducer.fit_transform(X_scaled)

df_s3["umap1"] = embedding[:, 0]
df_s3["umap2"] = embedding[:, 1]

# =====================
# 16. PLOTS
# =====================

# --- UMAP by cluster ---
plt.figure(figsize=(6,5))
sns.scatterplot(data=df_s3, x="umap1", y="umap2", hue="cluster", s=20)
plt.title("Scenario 3 — UMAP by physiological cluster")
plt.tight_layout()
plt.show()

# --- UMAP by outcome ---
plt.figure(figsize=(6,5))
sns.scatterplot(data=df_s3, x="umap1", y="umap2", hue=outcome, s=20)
plt.title("Scenario 3 — UMAP by procurement outcome")
plt.tight_layout()
plt.show()

# --- Barplot procurement rate ---
proc_rates_df = df_s3.groupby("cluster")[outcome].mean().reset_index()
proc_rates_df["percent"] = proc_rates_df[outcome] * 100

plt.figure(figsize=(5,4))
sns.barplot(data=proc_rates_df, x="cluster", y="percent")
plt.ylabel("Procurement rate (%)")
plt.xlabel("Cluster")
plt.title("Scenario 3 — Procurement rate by cluster")
plt.tight_layout()
plt.show()

# --- Feature importance plot ---
plt.figure(figsize=(6,4))
sns.barplot(x=importance.values[:10], y=importance.index[:10])
plt.title("Scenario 3 — Top variables defining clusters")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# =====================
# END
# =====================

In [ ]:
# ============================================================
# SCENARIO 3 — BD donors, outcome = procurement (FULL PIPELINE)
# ============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

import statsmodels.api as sm
import umap.umap_ as umap

# =====================
# 1. FILTER
# =====================
df_s3 = df[df["brain_death"] == True].copy()

print("Scenario 3 N =", len(df_s3))

# =====================
# 2. OUTCOME
# =====================
outcome = "procured"
df_s3[outcome] = df_s3[outcome].astype(int)

# =====================
# 3. FEATURES — PHYSIOLOGY ONLY
# =====================
candidate_features = [
    "age", "weight_kg", "height_in",
    "chem_creatinine_mean", "chem_sodium_mean", "chem_potassium_mean",
    "cbc_wbc_mean", "cbc_hgb_mean",
    "abg_ph_mean", "abg_pco2_mean", "abg_po2_mean",
    "hemo_heartrate_mean", "hemo_bpsystolic_mean", "hemo_bpdiastolic_mean",
    "total_intake", "total_output"
]

features = [f for f in candidate_features if f in df_s3.columns]

# =====================
# 4. MISSING FILTER
# =====================
missing = df_s3[features].isna().mean()
features = missing[missing < 0.4].index.tolist()

print("\nFeatures after missing filter:")
print(features)

# =====================
# 5. IMPUTATION
# =====================
X = df_s3[features].copy() # Ensure X has the original index

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=features, index=X.index)

# =====================
# 6. VARIANCE FILTER
# =====================
selector = VarianceThreshold(threshold=0.01)
X_var_array = selector.fit_transform(X_imp)

features_var = np.array(features)[selector.get_support()]
X_var = pd.DataFrame(X_var_array, columns=features_var, index=X_imp.index) # Preserve index

# =====================
# 7. CORRELATION FILTER
# =====================
corr = X_var.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
X_final = X_var.drop(columns=to_drop) # This will naturally preserve the index

print("\nFinal features:")
print(X_final.columns.tolist())

# =====================
# 8. STANDARDIZE
# =====================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# =====================
# 9. CLUSTER SELECTION
# =====================
sil_scores = []

for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    sil_scores.append((k, silhouette_score(X_scaled, labels)))

print("\nSilhouette scores:", sil_scores)

# Forçar interpretabilidade
k_opt = 2

# =====================
# 10. FINAL CLUSTERING
# =====================
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=20)
df_s3["cluster"] = kmeans.fit_predict(X_scaled)

print("\nCluster distribution:")
print(df_s3["cluster"].value_counts())

# =====================
# 11. CLUSTER DESCRIPTION
# =====================
cluster_means = df_s3.groupby("cluster")[X_final.columns].mean()

print("\nCluster means:")
print(cluster_means)

# =====================
# 12. FEATURE IMPORTANCE
# =====================
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_scaled, df_s3["cluster"])

importance = pd.Series(rf.feature_importances_, index=X_final.columns)
importance = importance.sort_values(ascending=False)

print("\nTop variables defining clusters:")
print(importance.head(10))

# =====================
# 13. OUTCOME BY CLUSTER
# =====================
proc_rates = df_s3.groupby("cluster")[outcome].mean()

print("\nProcurement rate by cluster:")
print(proc_rates)

# =====================
# 14. LOGISTIC REGRESSION
# =====================
X_model = sm.add_constant(df_s3[["cluster", "age"]])
y = df_s3[outcome]

model = sm.Logit(y, X_model).fit()

print("\nLogistic regression:")
print(model.summary())

# =====================
# 15. UMAP
# =====================
reducer = umap.UMAP(random_state=42)
embedding = reducer.fit_transform(X_scaled)

df_s3["umap1"] = embedding[:, 0]
df_s3["umap2"] = embedding[:, 1]

# =====================
# 16. PLOTS
# =====================

# --- UMAP by cluster ---
plt.figure(figsize=(6,5))
sns.scatterplot(data=df_s3, x="umap1", y="umap2", hue="cluster", s=20)
plt.title("Scenario 3 — UMAP by physiological cluster")
plt.tight_layout()
plt.show()

# --- UMAP by outcome ---
plt.figure(figsize=(6,5))
sns.scatterplot(data=df_s3, x="umap1", y="umap2", hue=outcome, s=20)
plt.title("Scenario 3 — UMAP by procurement outcome")
plt.tight_layout()
plt.show()

# --- Barplot procurement rate ---
proc_rates_df = df_s3.groupby("cluster")[outcome].mean().reset_index()
proc_rates_df["percent"] = proc_rates_df[outcome] * 100

plt.figure(figsize=(5,4))
sns.barplot(data=proc_rates_df, x="cluster", y="percent")
plt.ylabel("Procurement rate (%)")
plt.xlabel("Cluster")
plt.title("Scenario 3 — Procurement rate by cluster")
plt.tight_layout()
plt.show()

# --- Feature importance plot ---
plt.figure(figsize=(6,4))
sns.barplot(x=importance.values[:10], y=importance.index[:10])
plt.title("Scenario 3 — Top variables defining clusters")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# =====================
# END
# =====================

In [ ]:
# ============================================================
# SCENARIO 3 — EFFECT OF OPO AFTER PHYSIOLOGICAL CLUSTERING
# BD donors | outcome = procured
# Assumes df_s3 and df_s3["cluster"] already exist
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# -----------------------------
# 0. COPY + BASIC PREPARATION
# -----------------------------
df_opo = df_s3.copy()

outcome = "procured"
df_opo[outcome] = df_opo[outcome].astype(int)

df_opo["cluster"] = df_opo["cluster"].astype("category")
df_opo["opo"] = df_opo["opo"].astype("category")

df_opo = df_opo.dropna(subset=["opo", "cluster", "age", outcome]).copy()

print("N for OPO analysis =", len(df_opo))
print("Number of clusters =", df_opo["cluster"].nunique())
print("Number of OPOs =", df_opo["opo"].nunique())

# -----------------------------
# 1. DESCRIPTIVE TABLES
# -----------------------------
print("\nPROCUREMENT RATE BY CLUSTER")
cluster_rates = df_opo.groupby("cluster", observed=False)[outcome].agg(["mean", "sum", "count"])
cluster_rates["percent"] = cluster_rates["mean"] * 100
print(cluster_rates)

print("\nPROCUREMENT RATE BY OPO")
opo_rates = (
    df_opo.groupby("opo", observed=False)[outcome]
    .agg(["mean", "sum", "count"])
    .sort_values("mean", ascending=False)
)
opo_rates["percent"] = opo_rates["mean"] * 100
print(opo_rates)

print("\nPROCUREMENT RATE BY CLUSTER × OPO")
cluster_opo_table = df_opo.pivot_table(
    index="cluster",
    columns="opo",
    values=outcome,
    aggfunc=["mean", "count"],
    observed=False
)
print(cluster_opo_table)

# -----------------------------
# 2. HEATMAP OF OBSERVED RATES
# -----------------------------
pivot_rate = df_opo.pivot_table(
    index="cluster",
    columns="opo",
    values=outcome,
    aggfunc="mean",
    observed=False
)

plt.figure(figsize=(max(10, pivot_rate.shape[1] * 0.6), 5))
sns.heatmap(pivot_rate, annot=True, fmt=".2f", cmap="gray")
plt.title("Procurement rate by physiological cluster and OPO — Scenario 3")
plt.xlabel("OPO")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

# -----------------------------
# 3. FILTER SMALL OPOs
# -----------------------------
min_n_opo = 10
valid_opos = df_opo["opo"].value_counts()
valid_opos = valid_opos[valid_opos >= min_n_opo].index

df_opo_filt = df_opo[df_opo["opo"].isin(valid_opos)].copy()
df_opo_filt["opo"] = df_opo_filt["opo"].astype("category")
df_opo_filt["cluster"] = df_opo_filt["cluster"].astype("category")

print("\nAfter filtering OPOs >=", min_n_opo)
print("N =", len(df_opo_filt))
print("OPOs kept =", df_opo_filt["opo"].nunique())

# -----------------------------
# 4. LOGISTIC MODEL:
#    cluster + opo + age
# -----------------------------
print("\nMODEL 1: procured ~ cluster + opo + age")

formula_main = f"{outcome} ~ C(cluster) + C(opo) + age"
model_main = smf.logit(formula=formula_main, data=df_opo_filt).fit(disp=False, maxiter=200)
print(model_main.summary())

or_main = pd.DataFrame({
    "OR": np.exp(model_main.params),
    "CI_lower": np.exp(model_main.conf_int()[0]),
    "CI_upper": np.exp(model_main.conf_int()[1]),
    "p_value": model_main.pvalues
}).sort_values("p_value")

print("\nOdds ratios:")
print(or_main)

# -----------------------------
# 5. INTERACTION MODEL:
#    cluster * opo + age
# -----------------------------
print("\nMODEL 2: procured ~ cluster * opo + age")

formula_int = f"{outcome} ~ C(cluster) * C(opo) + age"

try:
    model_int = smf.logit(formula=formula_int, data=df_opo_filt).fit(disp=False, maxiter=300)
    print(model_int.summary())

    lr_stat = 2 * (model_int.llf - model_main.llf)
    df_diff = model_int.df_model - model_main.df_model
    lr_p = chi2.sf(lr_stat, df_diff)

    print("\nInteraction test:")
    print(f"LR = {lr_stat:.3f} | df = {df_diff} | p = {lr_p:.5f}")

    interaction_success = True

except Exception as e:
    print("Interaction model failed:", e)
    interaction_success = False

# -----------------------------
# 6. WITHIN-CLUSTER OPO EFFECT
# -----------------------------
print("\nWITHIN-CLUSTER OPO EFFECT")

results = []

for cl in df_opo_filt["cluster"].cat.categories:
    sub = df_opo_filt[df_opo_filt["cluster"] == cl].copy()

    if len(sub) < 20:
        print(f"\nCluster {cl}: skipped (n < 20)")
        continue

    valid_sub_opos = sub["opo"].value_counts()
    valid_sub_opos = valid_sub_opos[valid_sub_opos >= 5].index
    sub = sub[sub["opo"].isin(valid_sub_opos)].copy()

    if sub["opo"].nunique() < 2:
        print(f"\nCluster {cl}: skipped (not enough opo variation)")
        continue

    print(f"\nCluster {cl}")
    print(sub.groupby("opo", observed=False)[outcome].agg(["mean", "sum", "count"]).sort_values("mean", ascending=False))

    try:
        m = smf.logit(f"{outcome} ~ C(opo) + age", data=sub).fit(disp=False, maxiter=200)
        null = smf.logit(f"{outcome} ~ age", data=sub).fit(disp=False, maxiter=200)

        lr = 2 * (m.llf - null.llf)
        p = chi2.sf(lr, m.df_model - null.df_model)

        results.append({
            "cluster": cl,
            "n": len(sub),
            "n_opo": sub["opo"].nunique(),
            "LR_stat": lr,
            "p_value": p
        })

        print("OPO effect p =", p)

    except Exception as e:
        print("Model failed in cluster", cl, ":", e)

results = pd.DataFrame(results)
print("\nSummary:")
print(results)

# -----------------------------
# 7. PREDICTED PROBABILITIES
# -----------------------------
mean_age = df_opo_filt["age"].mean()

grid = pd.DataFrame([
    {"cluster": cl, "opo": opo, "age": mean_age}
    for cl in df_opo_filt["cluster"].cat.categories
    for opo in df_opo_filt["opo"].cat.categories
])

grid["pred"] = model_main.predict(grid)

pivot_pred = grid.pivot(index="cluster", columns="opo", values="pred")

print("\nPredicted probabilities (adjusted):")
print(pivot_pred)

plt.figure(figsize=(max(10, pivot_pred.shape[1] * 0.6), 5))
sns.heatmap(pivot_pred, annot=True, fmt=".2f", cmap="gray")
plt.title("Predicted probability of procurement (adjusted) — Scenario 3")
plt.xlabel("OPO")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

# -----------------------------
# 8. OBSERVED BARPLOT
# -----------------------------
plt.figure(figsize=(12, 6))
sns.barplot(data=df_opo_filt, x="cluster", y=outcome, hue="opo", errorbar=None)
plt.title("Observed procurement rate by cluster and OPO — Scenario 3")
plt.ylabel("Procurement rate")
plt.tight_layout()
plt.show()

# -----------------------------
# 9. OPTIONAL GLOBAL TEST FOR OPO
#    Compare age+cluster vs age+cluster+opo
# -----------------------------
print("\nGLOBAL TEST OF OPO EFFECT")

model_no_opo = smf.logit(f"{outcome} ~ C(cluster) + age", data=df_opo_filt).fit(disp=False, maxiter=200)

lr_opo = 2 * (model_main.llf - model_no_opo.llf)
df_opo_test = model_main.df_model - model_no_opo.df_model
p_opo_global = chi2.sf(lr_opo, df_opo_test)

print(f"LR = {lr_opo:.3f} | df = {df_opo_test} | p = {p_opo_global:.5f}")

# -----------------------------
# 10. SAVE OUTPUTS
# -----------------------------
cluster_rates.to_excel("scenario3_cluster_rates.xlsx")
opo_rates.to_excel("scenario3_opo_rates.xlsx")
cluster_opo_table.to_excel("scenario3_cluster_opo.xlsx")
pivot_pred.to_excel("scenario3_predicted_cluster_opo.xlsx")
results.to_excel("scenario3_within_cluster_opo.xlsx")
or_main.to_excel("scenario3_or_main_model.xlsx")

print("\nFiles saved:")
print("- scenario3_cluster_rates.xlsx")
print("- scenario3_opo_rates.xlsx")
print("- scenario3_cluster_opo.xlsx")
print("- scenario3_predicted_cluster_opo.xlsx")
print("- scenario3_within_cluster_opo.xlsx")
print("- scenario3_or_main_model.xlsx")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Features e outcome
X = X_final   # Use the already imputed and processed X_final from Scenario 3
y = df_s3["procured"]

# Modelo
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Probabilidades
y_prob = model.predict_proba(X)[:, 1]

# ROC
fpr, tpr, thresholds = roc_curve(y, y_prob)
roc_auc = auc(fpr, tpr)

print("AUC:", roc_auc)

# Plotting the ROC curve
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle="--", linewidth=1)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Procurement Prediction (Scenario 3)")

plt.legend(loc="lower right")
plt.grid(alpha=0.2)

plt.show()

 PIPELINE SCENARIO 4

Non Brain-dead donors.
Outcome: Procured versus not procured.
Variaveis fisiológicas e variáveis processo.

In [ ]:
# ============================================================
# SCENARIO 4 — non-BD donors, outcome = procurement (FULL PIPELINE)
# ============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

import statsmodels.api as sm
import umap.umap_ as umap

# =====================
# 1. FILTER
# =====================
df_s4 = df[df["brain_death"] == False].copy()

print("Scenario 4 N =", len(df_s4))

# =====================
# 2. OUTCOME
# =====================
outcome = "procured"
df_s4[outcome] = df_s4[outcome].astype(int)

# =====================
# 3. FEATURES — PHYSIOLOGY ONLY
# =====================
candidate_features = [
    "age", "weight_kg", "height_in",
    "chem_creatinine_mean", "chem_sodium_mean", "chem_potassium_mean",
    "cbc_wbc_mean", "cbc_hgb_mean",
    "abg_ph_mean", "abg_pco2_mean", "abg_po2_mean",
    "hemo_heartrate_mean", "hemo_bpsystolic_mean", "hemo_bpdiastolic_mean",
    "total_intake", "total_output"
]

features = [f for f in candidate_features if f in df_s4.columns]

# =====================
# 4. MISSING FILTER
# =====================
missing = df_s4[features].isna().mean()
features = missing[missing < 0.4].index.tolist()

print("\nFeatures after missing filter:")
print(features)

# =====================
# 5. IMPUTATION
# =====================
X = df_s4[features].copy()

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=features, index=X.index)

# =====================
# 6. VARIANCE FILTER
# =====================
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X_imp)

features_var = np.array(features)[selector.get_support()]
X_var = pd.DataFrame(X_var, columns=features_var, index=X_imp.index)

# =====================
# 7. CORRELATION FILTER
# =====================
corr = X_var.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
X_final = X_var.drop(columns=to_drop)

print("\nFinal features:")
print(X_final.columns.tolist())

# =====================
# 8. STANDARDIZE
# =====================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# =====================
# 9. CLUSTER SELECTION
# =====================
sil_scores = []

for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    sil_scores.append((k, silhouette_score(X_scaled, labels)))

print("\nSilhouette scores:", sil_scores)

# Force interpretability
k_opt = 2

# =====================
# 10. FINAL CLUSTERING
# =====================
kmeans = KMeans(n_clusters=k_opt, random_state=42, n_init=20)
df_s4["cluster"] = kmeans.fit_predict(X_scaled)

print("\nCluster distribution:")
print(df_s4["cluster"].value_counts())

# =====================
# 11. CLUSTER DESCRIPTION
# =====================
cluster_means = df_s4.groupby("cluster")[X_final.columns].mean()

print("\nCluster means:")
print(cluster_means)

# =====================
# 12. FEATURE IMPORTANCE
# =====================
rf = RandomForestClassifier(n_estimators=500, random_state=42)
rf.fit(X_scaled, df_s4["cluster"])

importance = pd.Series(rf.feature_importances_, index=X_final.columns).sort_values(ascending=False)

print("\nTop variables defining clusters:")
print(importance.head(10))

# =====================
# 13. OUTCOME BY CLUSTER
# =====================
proc_rates = df_s4.groupby("cluster")[outcome].mean()

print("\nProcurement rate by cluster:")
print(proc_rates)

# =====================
# 14. LOGISTIC REGRESSION
# =====================
X_model = sm.add_constant(df_s4[["cluster", "age"]])
y = df_s4[outcome]

model = sm.Logit(y, X_model).fit()

print("\nLogistic regression:")
print(model.summary())

# Optional OR table
or_table = pd.DataFrame({
    "OR": np.exp(model.params),
    "CI_low": np.exp(model.conf_int()[0]),
    "CI_high": np.exp(model.conf_int()[1]),
    "p": model.pvalues
})
print("\nOdds ratios:")
print(or_table)

# =====================
# 15. UMAP
# =====================
reducer = umap.UMAP(random_state=42)
embedding = reducer.fit_transform(X_scaled)

df_s4["umap1"] = embedding[:, 0]
df_s4["umap2"] = embedding[:, 1]

# =====================
# 16. PLOTS
# =====================

# --- UMAP by cluster ---
plt.figure(figsize=(6,5))
sns.scatterplot(data=df_s4, x="umap1", y="umap2", hue="cluster", s=20)
plt.title("Scenario 4 — UMAP by physiological cluster")
plt.tight_layout()
plt.show()

# --- UMAP by outcome ---
plt.figure(figsize=(6,5))
sns.scatterplot(data=df_s4, x="umap1", y="umap2", hue=outcome, s=20)
plt.title("Scenario 4 — UMAP by procurement outcome")
plt.tight_layout()
plt.show()

# --- Barplot procurement rate ---
proc_rates_df = df_s4.groupby("cluster")[outcome].mean().reset_index()
proc_rates_df["percent"] = proc_rates_df[outcome] * 100

plt.figure(figsize=(5,4))
sns.barplot(data=proc_rates_df, x="cluster", y="percent")
plt.ylabel("Procurement rate (%)")
plt.xlabel("Cluster")
plt.title("Scenario 4 — Procurement rate by cluster")
plt.tight_layout()
plt.show()

# --- Feature importance plot ---
plt.figure(figsize=(6,4))
sns.barplot(x=importance.values[:10], y=importance.index[:10])
plt.title("Scenario 4 — Top variables defining clusters")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# =====================
# END
# =====================

In [ ]:
# ============================================================
# SCENARIO 4 — EFFECT OF OPO AFTER PHYSIOLOGICAL CLUSTERING
# non-BD donors | outcome = procured
# Assumes df_s4 and df_s4["cluster"] already exist
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# -----------------------------
# 0. COPY + BASIC PREPARATION
# -----------------------------
df_opo = df_s4.copy()

outcome = "procured"
df_opo[outcome] = df_opo[outcome].astype(int)

df_opo["cluster"] = df_opo["cluster"].astype("category")
df_opo["opo"] = df_opo["opo"].astype("category")

df_opo = df_opo.dropna(subset=["opo", "cluster", "age", outcome]).copy()

print("N for OPO analysis =", len(df_opo))
print("Number of clusters =", df_opo["cluster"].nunique())
print("Number of OPOs =", df_opo["opo"].nunique())

# -----------------------------
# 1. DESCRIPTIVE TABLES
# -----------------------------
print("\nPROCUREMENT RATE BY CLUSTER")
cluster_rates = df_opo.groupby("cluster", observed=False)[outcome].agg(["mean", "sum", "count"])
cluster_rates["percent"] = cluster_rates["mean"] * 100
print(cluster_rates)

print("\nPROCUREMENT RATE BY OPO")
opo_rates = (
    df_opo.groupby("opo", observed=False)[outcome]
    .agg(["mean", "sum", "count"])
    .sort_values("mean", ascending=False)
)
opo_rates["percent"] = opo_rates["mean"] * 100
print(opo_rates)

print("\nPROCUREMENT RATE BY CLUSTER × OPO")
cluster_opo_table = df_opo.pivot_table(
    index="cluster",
    columns="opo",
    values=outcome,
    aggfunc=["mean", "count"],
    observed=False
)
print(cluster_opo_table)

# -----------------------------
# 2. HEATMAP OF OBSERVED RATES
# -----------------------------
pivot_rate = df_opo.pivot_table(
    index="cluster",
    columns="opo",
    values=outcome,
    aggfunc="mean",
    observed=False
)

plt.figure(figsize=(max(10, pivot_rate.shape[1] * 0.6), 5))
sns.heatmap(pivot_rate, annot=True, fmt=".2f", cmap="gray")
plt.title("Procurement rate by physiological cluster and OPO — Scenario 4")
plt.xlabel("OPO")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

# -----------------------------
# 3. FILTER SMALL OPOs
# -----------------------------
min_n_opo = 10
valid_opos = df_opo["opo"].value_counts()
valid_opos = valid_opos[valid_opos >= min_n_opo].index

df_opo_filt = df_opo[df_opo["opo"].isin(valid_opos)].copy()
df_opo_filt["opo"] = df_opo_filt["opo"].astype("category")
df_opo_filt["cluster"] = df_opo_filt["cluster"].astype("category")

print("\nAfter filtering OPOs >=", min_n_opo)
print("N =", len(df_opo_filt))
print("OPOs kept =", df_opo_filt["opo"].nunique())

# -----------------------------
# 4. LOGISTIC MODEL:
#    cluster + opo + age
# -----------------------------
print("\nMODEL 1: procured ~ cluster + opo + age")

formula_main = f"{outcome} ~ C(cluster) + C(opo) + age"
model_main = smf.logit(formula=formula_main, data=df_opo_filt).fit(disp=False, maxiter=200)
print(model_main.summary())

or_main = pd.DataFrame({
    "OR": np.exp(model_main.params),
    "CI_lower": np.exp(model_main.conf_int()[0]),
    "CI_upper": np.exp(model_main.conf_int()[1]),
    "p_value": model_main.pvalues
}).sort_values("p_value")

print("\nOdds ratios:")
print(or_main)

# -----------------------------
# 5. INTERACTION MODEL:
#    cluster * opo + age
# -----------------------------
print("\nMODEL 2: procured ~ cluster * opo + age")

formula_int = f"{outcome} ~ C(cluster) * C(opo) + age"

try:
    model_int = smf.logit(formula=formula_int, data=df_opo_filt).fit(disp=False, maxiter=300)
    print(model_int.summary())

    lr_stat = 2 * (model_int.llf - model_main.llf)
    df_diff = model_int.df_model - model_main.df_model
    lr_p = chi2.sf(lr_stat, df_diff)

    print("\nInteraction test:")
    print(f"LR = {lr_stat:.3f} | df = {df_diff} | p = {lr_p:.5f}")

    interaction_success = True

except Exception as e:
    print("Interaction model failed:", e)
    interaction_success = False

# -----------------------------
# 6. WITHIN-CLUSTER OPO EFFECT
# -----------------------------
print("\nWITHIN-CLUSTER OPO EFFECT")

results = []

for cl in df_opo_filt["cluster"].cat.categories:
    sub = df_opo_filt[df_opo_filt["cluster"] == cl].copy()

    if len(sub) < 20:
        print(f"\nCluster {cl}: skipped (n < 20)")
        continue

    valid_sub_opos = sub["opo"].value_counts()
    valid_sub_opos = valid_sub_opos[valid_sub_opos >= 5].index
    sub = sub[sub["opo"].isin(valid_sub_opos)].copy()

    if sub["opo"].nunique() < 2:
        print(f"\nCluster {cl}: skipped (not enough opo variation)")
        continue

    print(f"\nCluster {cl}")
    print(sub.groupby("opo", observed=False)[outcome].agg(["mean", "sum", "count"]).sort_values("mean", ascending=False))

    try:
        m = smf.logit(f"{outcome} ~ C(opo) + age", data=sub).fit(disp=False, maxiter=200)
        null = smf.logit(f"{outcome} ~ age", data=sub).fit(disp=False, maxiter=200)

        lr = 2 * (m.llf - null.llf)
        p = chi2.sf(lr, m.df_model - null.df_model)

        results.append({
            "cluster": cl,
            "n": len(sub),
            "n_opo": sub["opo"].nunique(),
            "LR_stat": lr,
            "p_value": p
        })

        print("OPO effect p =", p)

    except Exception as e:
        print("Model failed in cluster", cl, ":", e)

results = pd.DataFrame(results)
print("\nSummary:")
print(results)

# -----------------------------
# 7. PREDICTED PROBABILITIES
# -----------------------------
mean_age = df_opo_filt["age"].mean()

grid = pd.DataFrame([
    {"cluster": cl, "opo": opo, "age": mean_age}
    for cl in df_opo_filt["cluster"].cat.categories
    for opo in df_opo_filt["opo"].cat.categories
])

grid["pred"] = model_main.predict(grid)

pivot_pred = grid.pivot(index="cluster", columns="opo", values="pred")

print("\nPredicted probabilities (adjusted):")
print(pivot_pred)

plt.figure(figsize=(max(10, pivot_pred.shape[1] * 0.6), 5))
sns.heatmap(pivot_pred, annot=True, fmt=".2f", cmap="gray")
plt.title("Predicted probability of procurement (adjusted) — Scenario 4")
plt.xlabel("OPO")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

# -----------------------------
# 8. OBSERVED BARPLOT
# -----------------------------
plt.figure(figsize=(12, 6))
sns.barplot(data=df_opo_filt, x="cluster", y=outcome, hue="opo", errorbar=None)
plt.title("Observed procurement rate by cluster and OPO — Scenario 4")
plt.ylabel("Procurement rate")
plt.tight_layout()
plt.show()

# -----------------------------
# 9. OPTIONAL GLOBAL TEST FOR OPO
#    Compare age+cluster vs age+cluster+opo
# -----------------------------
print("\nGLOBAL TEST OF OPO EFFECT")

model_no_opo = smf.logit(f"{outcome} ~ C(cluster) + age", data=df_opo_filt).fit(disp=False, maxiter=200)

lr_opo = 2 * (model_main.llf - model_no_opo.llf)
df_opo_test = model_main.df_model - model_no_opo.df_model
p_opo_global = chi2.sf(lr_opo, df_opo_test)

print(f"LR = {lr_opo:.3f} | df = {df_opo_test} | p = {p_opo_global:.5f}")

# -----------------------------
# 10. SAVE OUTPUTS
# -----------------------------
cluster_rates.to_excel("scenario4_cluster_rates.xlsx")
opo_rates.to_excel("scenario4_opo_rates.xlsx")
cluster_opo_table.to_excel("scenario4_cluster_opo.xlsx")
pivot_pred.to_excel("scenario4_predicted_cluster_opo.xlsx")
results.to_excel("scenario4_within_cluster_opo.xlsx")
or_main.to_excel("scenario4_or_main_model.xlsx")

print("\nFiles saved:")
print("- scenario4_cluster_rates.xlsx")
print("- scenario4_opo_rates.xlsx")
print("- scenario4_cluster_opo.xlsx")
print("- scenario4_predicted_cluster_opo.xlsx")
print("- scenario4_within_cluster_opo.xlsx")
print("- scenario4_or_main_model.xlsx")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Filter for Scenario 4 data (non-BD donors)
df_s4_temp = df[df["brain_death"] == False].copy()

# Define outcome
outcome = "procured"
df_s4_temp[outcome] = df_s4_temp[outcome].astype(int)

# Define candidate features (physiology only)
candidate_features = [
    "age", "weight_kg", "height_in",
    "chem_creatinine_mean", "chem_sodium_mean", "chem_potassium_mean",
    "cbc_wbc_mean", "cbc_hgb_mean",
    "abg_ph_mean", "abg_pco2_mean", "abg_po2_mean",
    "hemo_heartrate_mean", "hemo_bpsystolic_mean", "hemo_bpdiastolic_mean",
    "total_intake", "total_output"
]
features = [f for f in candidate_features if f in df_s4_temp.columns]

# Missing filter
missing = df_s4_temp[features].isna().mean()
features = missing[missing < 0.4].index.tolist()

# Imputation
X = df_s4_temp[features].copy()
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=features, index=X.index)

# Variance filter
selector = VarianceThreshold(threshold=0.01)
X_var_array = selector.fit_transform(X_imp)
features_var = np.array(features)[selector.get_support()]
X_var = pd.DataFrame(X_var_array, columns=features_var, index=X_imp.index)

# Correlation filter
corr = X_var.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
X_final_s4 = X_var.drop(columns=to_drop)

# Standardize
scaler = StandardScaler()
X_scaled_s4 = scaler.fit_transform(X_final_s4)

# Features and outcome for Logistic Regression
X = X_scaled_s4 # Use the scaled features for Scenario 4
y = df_s4_temp["procured"]

# Model
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Probabilities
y_prob = model.predict_proba(X)[:, 1]

# ROC
fpr, tpr, thresholds = roc_curve(y, y_prob)
roc_auc = auc(fpr, tpr)

print("AUC:", roc_auc)

# Plotting the ROC curve
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle="--", linewidth=1)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Procurement Prediction (Scenario 4)")

plt.legend(loc="lower right")
plt.grid(alpha=0.2)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

# ================================
# HELPER FUNCTION: Prepare Features for Logistic Regression
# ================================
def prepare_features(df, scenario_name, outcome_col):
    df = df.copy()

    # Outcome
    y = df[outcome_col].astype(int)

    # Cluster as categorical
    if "cluster" not in df.columns:
        print(f"Error in {scenario_name}: 'cluster' column not found in DataFrame. Skipping feature preparation.")
        return None, None, None # X_clean, y_clean, feature_names

    X = pd.get_dummies(df["cluster"].astype(str), prefix="cluster", drop_first=True)

    # Add age
    if "age" in df.columns:
        X["age"] = df["age"]
    else:
        print(f"Warning in {scenario_name}: 'age' column not found for logistic regression. Skipping.")

    X = sm.add_constant(X)

    # Ensure all columns are numeric and handle NaNs
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
    y = pd.to_numeric(y, errors='coerce')

    # Combine X and y to drop rows with NaNs consistently
    data_for_logit = pd.concat([y.rename('outcome'), X], axis=1).dropna()
    y_clean = data_for_logit['outcome']
    X_clean = data_for_logit.drop(columns='outcome')

    if X_clean.empty or y_clean.empty:
        print(f"Warning in {scenario_name}: No valid data points after NaN handling. Skipping feature preparation.")
        return None, None, None

    if y_clean.nunique() < 2:
        print(f"Warning in {scenario_name}: Dependent variable '{outcome_col}' has no variation after cleaning. Skipping feature preparation.")
        return None, None, None

    return X_clean, y_clean, X_clean.columns.tolist()

# ================================
# FUNCTION: logistic regression (Generalized)
# ================================
def run_logistic_general(df, scenario_name, outcome_col):

    X_clean, y_clean, feature_names = prepare_features(df, scenario_name, outcome_col)

    if X_clean is None:
        return pd.DataFrame(), None, None, None # Empty OR table, no y_clean, X_clean, or model

    # Explicitly ensure dtypes are float after dropping NaNs to avoid statsmodels casting issues
    y_clean = y_clean.astype(float)
    X_clean = X_clean.astype(float)

    try:
        model = sm.Logit(y_clean, X_clean).fit(disp=0)
    except Exception as e:
        print(f"Error fitting Logit model for {scenario_name}: {e}")
        return pd.DataFrame(), None, None, None

    results = []
    param_names = model.params.index.tolist() if hasattr(model.params, 'index') else feature_names

    for var_name in param_names:
        if var_name == "const":
            continue

        try:
            coef = model.params[var_name]
            se = model.bse[var_name]
            p = model.pvalues[var_name]
        except KeyError:
            print(f"Warning in {scenario_name}: Could not find {var_name} in model parameters. Skipping.")
            continue

        OR = np.exp(coef)
        CI_low = np.exp(coef - 1.96 * se)
        CI_high = np.exp(coef + 1.96 * se)

        results.append({
            "Scenario": scenario_name,
            "Variable": var_name,
            "OR": round(OR, 3),
            "CI_low": round(CI_low, 3),
            "CI_high": round(CI_high, 3),
            "p_value": round(p, 4)
        })

    return pd.DataFrame(results), y_clean, X_clean, model

# ================================
# FUNCTION: Calculate AUC
# ================================
def calculate_auc(y_true, y_scores, scenario_name):
    if y_true is None or y_scores is None or len(y_true) == 0:
        return pd.DataFrame()

    try:
        auc_score = roc_auc_score(y_true, y_scores)
        return pd.DataFrame([{"Scenario": scenario_name, "AUC": round(auc_score, 3)}])
    except ValueError as e:
        # Handle cases where y_true contains only one class
        print(f"Error calculating AUC for {scenario_name}: {e}. This might happen if y_true contains only one class.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Unexpected error calculating AUC for {scenario_name}: {e}")
        return pd.DataFrame()

# ================================
# RUN FOR ALL SCENARIOS AND COMPILE RESULTS
# ================================
all_or_tables = []
all_auc_tables = []

scenarios_config = [
    (df_s1, "Scenario 1", "transplanted"),
    (df_s2, "Scenario 2", "transplanted"),
    (df_s3, "Scenario 3", "procured"),
    (df_s4, "Scenario 4", "procured"),
]

for df_scenario, scenario_name, outcome_col in scenarios_config:
    or_table_partial, y_clean, X_clean, model = run_logistic_general(df_scenario, scenario_name, outcome_col)
    if not or_table_partial.empty:
        all_or_tables.append(or_table_partial)
        if model: # Only calculate AUC if model fitting was successful
            y_prob = model.predict(X_clean)
            auc_table_partial = calculate_auc(y_clean, y_prob, scenario_name)
            if not auc_table_partial.empty:
                all_auc_tables.append(auc_table_partial)

final_or_table = pd.concat(all_or_tables, ignore_index=True)
final_auc_table = pd.concat(all_auc_tables, ignore_index=True)

# ================================
# FORMAT (paper-ready) OR Table
# ================================
if not final_or_table.empty:
    final_or_table["OR (95% CI)"] = final_or_table.apply(
        lambda x: f"{x['OR']} ({x['CI_low']}–{x['CI_high']})", axis=1
    )
    final_or_table = final_or_table[["Scenario", "Variable", "OR (95% CI)", "p_value"]]
    print("Odds Ratios and 95% Confidence Intervals:")
    display(final_or_table)
else:
    print("No Odds Ratios table generated due to errors.")

# ================================
# DISPLAY AUC Table
# ================================
if not final_auc_table.empty:
    print("\nAUC Scores:")
    display(final_auc_table)
else:
    print("No AUC scores generated due to errors.")

# ================================
# EXPORT
# ================================
if not final_or_table.empty:
    final_or_table.to_excel("Logistic_Regression_ORs.xlsx", index=False)
    final_or_table.to_csv("Logistic_Regression_ORs.csv", index=False)
    print("Saved: Logistic_Regression_ORs.xlsx and .csv")

if not final_auc_table.empty:
    final_auc_table.to_excel("Logistic_Regression_AUCs.xlsx", index=False)
    final_auc_table.to_csv("Logistic_Regression_AUCs.csv", index=False)
    print("Saved: Logistic_Regression_AUCs.xlsx and .csv")

In [ ]:
print("The logistic regression results have already been exported to 'Logistic_Regression_ORs.xlsx', 'Logistic_Regression_ORs.csv', 'Logistic_Regression_AUCs.xlsx', and 'Logistic_Regression_AUCs.csv' in the previous step.")

In [ ]:
# ============================================================
# FOREST PLOT — 4 SCENARIOS
# Uses existing dataframes: df_s1, df_s2, df_s3, df_s4
# Assumes each dataframe already contains:
#   - cluster
#   - age
#   - opo
#   - outcome column:
#       s1, s2 -> transplanted
#       s3, s4 -> procured
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
scenario_dict = {
    "Scenario 1": {"df": df_s1.copy(), "outcome": "transplanted"},
    "Scenario 2": {"df": df_s2.copy(), "outcome": "transplanted"},
    "Scenario 3": {"df": df_s3.copy(), "outcome": "procured"},
    "Scenario 4": {"df": df_s4.copy(), "outcome": "procured"},
}

AGE_COL = "age"
CLUSTER_COL = "cluster"
OPO_COL = "opo"

# mínimo por OPO para entrar no modelo
MIN_OPO_N = 10

# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------
def prepare_and_fit(data, outcome_col, scenario_name):
    d = data.copy()

    needed = [outcome_col, CLUSTER_COL, AGE_COL, OPO_COL]
    missing_cols = [c for c in needed if c not in d.columns]
    if missing_cols:
        raise ValueError(f"{scenario_name}: missing columns -> {missing_cols}")

    d = d[needed].dropna().copy()

    # garantir tipos
    d[outcome_col] = pd.to_numeric(d[outcome_col], errors="coerce")
    d[AGE_COL] = pd.to_numeric(d[AGE_COL], errors="coerce")
    d = d.dropna().copy()
    d = d[d[outcome_col].isin([0, 1])].copy()

    # categorias
    d[CLUSTER_COL] = d[CLUSTER_COL].astype("category")
    d[OPO_COL] = d[OPO_COL].astype("category")

    # filtrar OPOs pequenos
    opo_counts = d[OPO_COL].value_counts()
    keep_opos = opo_counts[opo_counts >= MIN_OPO_N].index
    d = d[d[OPO_COL].isin(keep_opos)].copy()

    d[CLUSTER_COL] = d[CLUSTER_COL].astype("category")
    d[OPO_COL] = d[OPO_COL].astype("category")
    d[CLUSTER_COL] = d[CLUSTER_COL].cat.remove_unused_categories()
    d[OPO_COL] = d[OPO_COL].cat.remove_unused_categories()

    if len(d) == 0:
        raise ValueError(f"{scenario_name}: no data left after cleaning")

    if d[outcome_col].nunique() < 2:
        raise ValueError(f"{scenario_name}: outcome has <2 classes after cleaning")

    if d[OPO_COL].nunique() < 2:
        raise ValueError(f"{scenario_name}: <2 OPO categories after filtering")

    if d[CLUSTER_COL].nunique() < 2:
        raise ValueError(f"{scenario_name}: <2 cluster categories")

    formula = f"{outcome_col} ~ C({CLUSTER_COL}) + {AGE_COL} + C({OPO_COL})"
    model = smf.logit(formula, data=d).fit(disp=False, maxiter=300)

    params = model.params
    conf = model.conf_int()
    pvals = model.pvalues

    out = pd.DataFrame({
        "term": params.index,
        "coef": params.values,
        "ci_low": conf[0].values,
        "ci_high": conf[1].values,
        "p": pvals.values
    })

    out = out[out["term"] != "Intercept"].copy()

    out["OR"] = np.exp(out["coef"])
    out["OR_low"] = np.exp(out["ci_low"])
    out["OR_high"] = np.exp(out["ci_high"])
    out["scenario"] = scenario_name

    def pretty_term(x):
        if x.startswith(f"C({CLUSTER_COL})"):
            return x.replace(f"C({CLUSTER_COL})[T.", "Cluster ").replace("]", "")
        elif x == AGE_COL:
            return "Age (per year)"
        elif x.startswith(f"C({OPO_COL})"):
            return x.replace(f"C({OPO_COL})[T.", "OPO ").replace("]", "")
        else:
            return x

    out["label"] = out["term"].apply(pretty_term)

    def sort_key(lbl):
        if str(lbl).startswith("Cluster"):
            return (0, str(lbl))
        elif lbl == "Age (per year)":
            return (1, str(lbl))
        elif str(lbl).startswith("OPO"):
            return (2, str(lbl))
        else:
            return (3, str(lbl))

    out = out.sort_values("label", key=lambda s: s.map(sort_key)).reset_index(drop=True)

    return d, model, out

# ------------------------------------------------------------
# FIT ALL SCENARIOS
# ------------------------------------------------------------
results = {}
models = {}
cleaned_data = {}

for sc, cfg in scenario_dict.items():
    try:
        d_clean, model, out = prepare_and_fit(cfg["df"], cfg["outcome"], sc)
        cleaned_data[sc] = d_clean
        models[sc] = model
        results[sc] = out
        print(f"{sc}: OK | n={len(d_clean)} | OPOs={d_clean[OPO_COL].nunique()} | clusters={d_clean[CLUSTER_COL].nunique()}")
    except Exception as e:
        print(f"{sc}: ERROR -> {e}")

# ------------------------------------------------------------
# FOREST PLOT
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(18, 13))
axes = axes.flatten()
scenario_order = ["Scenario 1", "Scenario 2", "Scenario 3", "Scenario 4"]

for ax, sc in zip(axes, scenario_order):
    if sc not in results:
        ax.axis("off")
        ax.set_title(f"{sc} (model failed)")
        continue

    res = results[sc].copy()

    y = np.arange(len(res))[::-1]
    xerr_low = res["OR"] - res["OR_low"]
    xerr_high = res["OR_high"] - res["OR"]

    ax.errorbar(
        res["OR"], y,
        xerr=[xerr_low, xerr_high],
        fmt='o',
        capsize=3,
        linewidth=1.4
    )

    ax.axvline(1, linestyle="--", linewidth=1)

    ax.set_yticks(y)
    ax.set_yticklabels(res["label"])
    ax.set_xscale("log")
    ax.set_xlabel("Adjusted odds ratio (log scale)")
    ax.set_title(sc)

    xmin = max(0.1, res["OR_low"].min() * 0.8)
    xmax = min(20, res["OR_high"].max() * 1.25)
    if xmin >= xmax:
        xmin, xmax = 0.25, 4
    ax.set_xlim(xmin, xmax)

    xr = ax.get_xlim()[1]
    for yi, (_, row) in zip(y, res.iterrows()):
        txt = f'{row["OR"]:.2f} ({row["OR_low"]:.2f}-{row["OR_high"]:.2f}), p={row["p"]:.3g}'
        ax.text(xr * 1.03, yi, txt, va="center", fontsize=8)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# COMBINED TABLE
# ------------------------------------------------------------
if len(results) > 0:
    forest_table = pd.concat(results.values(), ignore_index=True)
    forest_table = forest_table[["scenario", "label", "OR", "OR_low", "OR_high", "p"]].copy()
    forest_table["OR (95% CI)"] = forest_table.apply(
        lambda r: f'{r["OR"]:.2f} ({r["OR_low"]:.2f}-{r["OR_high"]:.2f})', axis=1
    )
    display(forest_table[["scenario", "label", "OR (95% CI)", "p"]])
else:
    print("No models were successfully fitted.")

In [ ]:
# ============================================================
# EFFECT COMPARISON PLOT — CLUSTER vs OPO
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
scenario_dict = {
    "Scenario 1": {"df": df_s1.copy(), "outcome": "transplanted"},
    "Scenario 2": {"df": df_s2.copy(), "outcome": "transplanted"},
    "Scenario 3": {"df": df_s3.copy(), "outcome": "procured"},
    "Scenario 4": {"df": df_s4.copy(), "outcome": "procured"},
}

AGE = "age"
CLUSTER = "cluster"
OPO = "opo"

results = []

# ------------------------------------------------------------
# LOOP
# ------------------------------------------------------------
for name, cfg in scenario_dict.items():

    d = cfg["df"].copy()
    outcome = cfg["outcome"]

    d = d[[outcome, CLUSTER, AGE, OPO]].dropna().copy()
    d = d[d[outcome].isin([0,1])]

    d[CLUSTER] = d[CLUSTER].astype("category")
    d[OPO] = d[OPO].astype("category")

    if d[CLUSTER].nunique() < 2 or d[OPO].nunique() < 2:
        continue

    # MODELS
    full = smf.logit(f"{outcome} ~ C({CLUSTER}) + {AGE} + C({OPO})", data=d).fit(disp=0)
    no_opo = smf.logit(f"{outcome} ~ C({CLUSTER}) + {AGE}", data=d).fit(disp=0)

    # -------------------------
    # CLUSTER EFFECT (OR)
    # -------------------------
    cluster_terms = [t for t in full.params.index if "cluster" in t.lower()]

    if len(cluster_terms) > 0:
        coef = full.params[cluster_terms[0]]
        OR = np.exp(coef)
    else:
        OR = np.nan

    # -------------------------
    # OPO EFFECT (LR statistic)
    # -------------------------
    LR = 2 * (full.llf - no_opo.llf)

    results.append({
        "Scenario": name,
        "Cluster_OR": OR,
        "OPO_LR": LR
    })

# ------------------------------------------------------------
# DATAFRAME
# ------------------------------------------------------------
df_plot = pd.DataFrame(results)

# normalizar LR para escala comparável (log transform)
df_plot["OPO_effect"] = np.log1p(df_plot["OPO_LR"])

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8,6))

y = np.arange(len(df_plot))

# cluster (log OR)
ax.scatter(np.log(df_plot["Cluster_OR"]), y, label="Cluster effect", s=80)

# OPO
ax.scatter(df_plot["OPO_effect"], y, label="OPO effect", s=80)

# labels
ax.set_yticks(y)
ax.set_yticklabels(df_plot["Scenario"])

ax.axvline(0, linestyle="--")

ax.set_xlabel("Effect size (log scale)")
ax.set_title("Relative contribution of physiology vs system")

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EFFECT COMPARISON PLOT — CLUSTER vs OPO
# Cluster = log(OR) with 95% CI
# OPO = log(1 + LR statistic) with bootstrap CI
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import chi2

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
scenario_dict = {
    "Scenario 1": {"df": df_s1.copy(), "outcome": "transplanted"},
    "Scenario 2": {"df": df_s2.copy(), "outcome": "transplanted"},
    "Scenario 3": {"df": df_s3.copy(), "outcome": "procured"},
    "Scenario 4": {"df": df_s4.copy(), "outcome": "procured"},
}

AGE = "age"
CLUSTER = "cluster"
OPO = "opo"
MIN_OPO_N = 10
N_BOOT = 200   # sobe para 500 se quiseres mais estável

# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------
def clean_data(d, outcome):
    d = d[[outcome, CLUSTER, AGE, OPO]].dropna().copy()
    d = d[d[outcome].isin([0, 1])].copy()
    d[CLUSTER] = d[CLUSTER].astype("category")
    d[OPO] = d[OPO].astype("category")

    opo_counts = d[OPO].value_counts()
    keep_opos = opo_counts[opo_counts >= MIN_OPO_N].index
    d = d[d[OPO].isin(keep_opos)].copy()

    d[CLUSTER] = d[CLUSTER].astype("category").cat.remove_unused_categories()
    d[OPO] = d[OPO].astype("category").cat.remove_unused_categories()
    return d

def fit_models(d, outcome):
    full = smf.logit(f"{outcome} ~ C({CLUSTER}) + {AGE} + C({OPO})", data=d).fit(disp=0, maxiter=300)
    no_opo = smf.logit(f"{outcome} ~ C({CLUSTER}) + {AGE}", data=d).fit(disp=0, maxiter=300)
    return full, no_opo

def bootstrap_opo_effect(d, outcome, n_boot=200, seed=42):
    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(n_boot):
        idx = rng.choice(d.index, size=len(d), replace=True)
        b = d.loc[idx].copy()

        # re-cast categories after bootstrap
        b[CLUSTER] = b[CLUSTER].astype("category").cat.remove_unused_categories()
        b[OPO] = b[OPO].astype("category").cat.remove_unused_categories()

        # need at least 2 levels
        if b[CLUSTER].nunique() < 2 or b[OPO].nunique() < 2 or b[outcome].nunique() < 2:
            continue

        try:
            full_b, no_opo_b = fit_models(b, outcome)
            lr_b = 2 * (full_b.llf - no_opo_b.llf)
            vals.append(np.log1p(lr_b))
        except:
            continue

    if len(vals) < 20:
        return np.nan, np.nan

    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
rows = []

for name, cfg in scenario_dict.items():
    d = clean_data(cfg["df"], cfg["outcome"])
    outcome = cfg["outcome"]

    if len(d) == 0 or d[CLUSTER].nunique() < 2 or d[OPO].nunique() < 2 or d[outcome].nunique() < 2:
        print(f"{name}: skipped")
        continue

    try:
        full, no_opo = fit_models(d, outcome)

        # cluster effect
        cluster_terms = [t for t in full.params.index if f"C({CLUSTER})" in t]
        if len(cluster_terms) == 0:
            print(f"{name}: no cluster term found")
            continue

        term = cluster_terms[0]
        coef = full.params[term]
        se = full.bse[term]

        cluster_log_or = coef
        cluster_low = coef - 1.96 * se
        cluster_high = coef + 1.96 * se

        # OPO effect
        lr = 2 * (full.llf - no_opo.llf)
        df_diff = full.df_model - no_opo.df_model
        p_opo = chi2.sf(lr, df_diff)
        opo_effect = np.log1p(lr)

        opo_low, opo_high = bootstrap_opo_effect(d, outcome, n_boot=N_BOOT, seed=42)

        rows.append({
            "Scenario": name,
            "cluster_effect": cluster_log_or,
            "cluster_low": cluster_low,
            "cluster_high": cluster_high,
            "opo_effect": opo_effect,
            "opo_low": opo_low,
            "opo_high": opo_high,
            "p_opo": p_opo
        })

        print(f"{name}: OK")

    except Exception as e:
        print(f"{name}: ERROR -> {e}")

plot_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6))

y = np.arange(len(plot_df))

# Cluster: square
ax.errorbar(
    plot_df["cluster_effect"],
    y,
    xerr=[
        plot_df["cluster_effect"] - plot_df["cluster_low"],
        plot_df["cluster_high"] - plot_df["cluster_effect"]
    ],
    fmt='s',
    markersize=7,
    capsize=4,
    linewidth=1.4,
    label="Cluster effect"
)

# OPO: triangle
if plot_df["opo_low"].notna().all():
    ax.errorbar(
        plot_df["opo_effect"],
        y,
        xerr=[
            plot_df["opo_effect"] - plot_df["opo_low"],
            plot_df["opo_high"] - plot_df["opo_effect"]
        ],
        fmt='^',
        markersize=8,
        capsize=4,
        linewidth=1.4,
        label="OPO effect"
    )
else:
    ax.scatter(
        plot_df["opo_effect"],
        y,
        marker='^',
        s=90,
        label="OPO effect"
    )

ax.axvline(0, linestyle="--", linewidth=1)

ax.set_yticks(y)
ax.set_yticklabels(plot_df["Scenario"])
ax.set_xlabel("Effect size (log scale)")
ax.set_title("Relative contribution of physiology versus system")

# annotate OPO p-values
xmax = max(plot_df["opo_effect"].max(), plot_df["cluster_high"].max()) + 0.25
for i, row in plot_df.iterrows():
    ax.text(xmax, i, f'OPO p={row["p_opo"]:.3g}', va="center", fontsize=9)

ax.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# TABLE
# ------------------------------------------------------------
display(plot_df.round(3))

In [ ]:
# ============================================================
# EFFECT COMPARISON PLOT — CLUSTER vs OPO
# Cluster = log(OR) with 95% CI
# OPO = log(1 + LR statistic) with bootstrap CI
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import chi2

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
scenario_dict = {
    "Scenario 4": {"df": df_s4.copy(), "outcome": "procured"},
    "Scenario 3": {"df": df_s3.copy(), "outcome": "procured"},
    "Scenario 2": {"df": df_s2.copy(), "outcome": "transplanted"},
    "Scenario 1": {"df": df_s1.copy(), "outcome": "transplanted"},
}

AGE = "age"
CLUSTER = "cluster"
OPO = "opo"
MIN_OPO_N = 10
N_BOOT = 200   # sobe para 500 se quiseres mais estável

# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------
def clean_data(d, outcome):
    d = d[[outcome, CLUSTER, AGE, OPO]].dropna().copy()
    d = d[d[outcome].isin([0, 1])].copy()
    d[CLUSTER] = d[CLUSTER].astype("category")
    d[OPO] = d[OPO].astype("category")

    opo_counts = d[OPO].value_counts()
    keep_opos = opo_counts[opo_counts >= MIN_OPO_N].index
    d = d[d[OPO].isin(keep_opos)].copy()

    d[CLUSTER] = d[CLUSTER].astype("category").cat.remove_unused_categories()
    d[OPO] = d[OPO].astype("category").cat.remove_unused_categories()
    return d

def fit_models(d, outcome):
    full = smf.logit(f"{outcome} ~ C({CLUSTER}) + {AGE} + C({OPO})", data=d).fit(disp=0, maxiter=300)
    no_opo = smf.logit(f"{outcome} ~ C({CLUSTER}) + {AGE}", data=d).fit(disp=0, maxiter=300)
    return full, no_opo

def bootstrap_opo_effect(d, outcome, n_boot=200, seed=42):
    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(n_boot):
        idx = rng.choice(d.index, size=len(d), replace=True)
        b = d.loc[idx].copy()

        # re-cast categories after bootstrap
        b[CLUSTER] = b[CLUSTER].astype("category").cat.remove_unused_categories()
        b[OPO] = b[OPO].astype("category").cat.remove_unused_categories()

        # need at least 2 levels
        if b[CLUSTER].nunique() < 2 or b[OPO].nunique() < 2 or b[outcome].nunique() < 2:
            continue

        try:
            full_b, no_opo_b = fit_models(b, outcome)
            lr_b = 2 * (full_b.llf - no_opo_b.llf)
            vals.append(np.log1p(lr_b))
        except:
            continue

    if len(vals) < 20:
        return np.nan, np.nan

    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
rows = []

for name, cfg in scenario_dict.items():
    d = clean_data(cfg["df"], cfg["outcome"])
    outcome = cfg["outcome"]

    if len(d) == 0 or d[CLUSTER].nunique() < 2 or d[OPO].nunique() < 2 or d[outcome].nunique() < 2:
        print(f"{name}: skipped")
        continue

    try:
        full, no_opo = fit_models(d, outcome)

        # cluster effect
        cluster_terms = [t for t in full.params.index if f"C({CLUSTER})" in t]
        if len(cluster_terms) == 0:
            print(f"{name}: no cluster term found")
            continue

        term = cluster_terms[0]
        coef = full.params[term]
        se = full.bse[term]

        cluster_log_or = coef
        cluster_low = coef - 1.96 * se
        cluster_high = coef + 1.96 * se

        # OPO effect
        lr = 2 * (full.llf - no_opo.llf)
        df_diff = full.df_model - no_opo.df_model
        p_opo = chi2.sf(lr, df_diff)
        opo_effect = np.log1p(lr)

        opo_low, opo_high = bootstrap_opo_effect(d, outcome, n_boot=N_BOOT, seed=42)

        rows.append({
            "Scenario": name,
            "cluster_effect": cluster_log_or,
            "cluster_low": cluster_low,
            "cluster_high": cluster_high,
            "opo_effect": opo_effect,
            "opo_low": opo_low,
            "opo_high": opo_high,
            "p_opo": p_opo
        })

        print(f"{name}: OK")

    except Exception as e:
        print(f"{name}: ERROR -> {e}")

plot_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6))

y = np.arange(len(plot_df))

# Cluster: square
ax.errorbar(
    plot_df["cluster_effect"],
    y,
    xerr=[
        plot_df["cluster_effect"] - plot_df["cluster_low"],
        plot_df["cluster_high"] - plot_df["cluster_effect"]
    ],
    fmt='s',
    markersize=7,
    capsize=4,
    linewidth=1.4,
    label="Cluster effect"
)

# OPO: triangle
if plot_df["opo_low"].notna().all():
    ax.errorbar(
        plot_df["opo_effect"],
        y,
        xerr=[
            plot_df["opo_effect"] - plot_df["opo_low"],
            plot_df["opo_high"] - plot_df["opo_effect"]
        ],
        fmt='^',
        markersize=8,
        capsize=4,
        linewidth=1.4,
        label="OPO effect"
    )
else:
    ax.scatter(
        plot_df["opo_effect"],
        y,
        marker='^',
        s=90,
        label="OPO effect"
    )

ax.axvline(0, linestyle="--", linewidth=1)

ax.set_yticks(y)
ax.set_yticklabels(plot_df["Scenario"])
ax.set_xlabel("Effect size (log scale)")
ax.set_title("Relative contribution of physiology versus system")

ax.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# TABLE
# ------------------------------------------------------------
display(plot_df.round(3))